# 🌾 Krishi-RAG — Agricultural QA Retrieval-Augmented Generation Core
### PATCHED for scalability/reliability (v2) — architecture unchanged

```
Farmer question -> optional translation -> query embedding -> Scalable FAISS
-> Top-K COMPLETE QA records -> Local instruction LLM -> Grounded synthesis
-> optional translation -> FINAL ANSWER
```

**This revision does not change the approved architecture.** It patches
engineering so the pipeline can run reliably on a multi-million-record
corpus on a single Colab T4 without RAM explosions, GPU-memory blowups,
O(n²) operations, or loss of progress after a disconnect.

**What changed vs the previous notebook (see chat message for full cell-by-cell audit):**
- FarmerChat is parsed with a true streaming reader (`ijson` for JSON arrays,
  line iteration for JSONL) and cleaned records are written straight to disk —
  no multi-million-item Python list is ever held in RAM.
- The final knowledge base lives on disk as `final_knowledge_base.jsonl`
  with a compact NumPy `int64` **byte-offset index** (`record_id -> file
  offset`). There is no giant in-memory `KB_RECORDS` list or `{id: {...}}`
  dict — `get_record(record_id)` seeks and reads exactly one line.
- Embeddings are generated in batches straight to a disk-backed memmap and
  are checkpointed; FarmerChat's massive scale never touches GPU memory as a
  whole, only one batch at a time.
- FAISS is built by adding **chunks** of the memmap (never the whole array
  at once), with periodic index checkpointing so a disconnect mid-build
  doesn't cost the whole build.
- FAISS insertion order == `record_id`, so there is no separate
  vector-id-to-record dict at all — the FAISS result position **is** the
  record id.
- Research ablations (Q-only vs Q+A embedding, Top-K sweep, generation-eval
  sweep, latency sweep) are OFF by default behind explicit flags — the first
  run builds the working core system only.
- A small (~300-record) end-to-end smoke test runs and must pass **before**
  full-corpus embedding starts.
- The old fake health check (`refused in (True, False)`) is replaced with a
  real refusal-behavior test on a deliberately nonsense query.


## SECTION 1 — Project Configuration

In [1]:

# ============================================================
# SECTION 1: PROJECT CONFIGURATION (single source of truth)
# ============================================================
import os, sys, json, time, hashlib, random, logging, traceback, gc, re, unicodedata, csv
from datetime import datetime
from pathlib import Path
from itertools import islice

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
SEED = 42

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "paths": {},   # filled in Section 4
    "datasets": {
        "dataset_26k_name": "dataset26k.json",
        "farmerchat_name": "farmerchat.json",   # file OR a directory of .json/.jsonl shards
    },
    "embedding": {
        "model_name": "BAAI/bge-m3",
        "normalize": True,
        "batch_size": 64,
        "max_seq_length": 512,
        "checkpoint_every_batches": 20,
        "representation": "question_only",   # production representation; fixed unless ablation says otherwise
    },
    "faiss": {
        "auto_select": True,
        "force_index_type": None,
        "add_chunk_size": 100_000,      # PATCH 3: chunked index.add()
        "save_every_n_chunks": 5,       # PATCH 16: periodic checkpoint during build
        "hnsw_m": 32, "hnsw_ef_construction": 200, "hnsw_ef_search": 128,
        "pq_m_divisor": 8, "pq_bits": 8, "nprobe": 16,
        "nlist_target_docs_per_cluster": 100,
    },
    "retrieval": {"top_k": 5, "min_score_for_confidence": 0.35},
    "generation": {
        # PATCH FINAL: each candidate declares whether 4-bit is REQUIRED (no unquantized
        # fallback allowed -- prevents the "silently loads 7B in fp16/fp32" failure mode)
        # and the minimum acceptable generation speed. The last candidate has
        # min_tokens_per_sec=0 -- it is the guaranteed-fast safety net and is always accepted.
        "candidate_models": [
            {"name": "Qwen/Qwen2.5-7B-Instruct", "require_4bit": True,  "min_tokens_per_sec": 8},
            {"name": "Qwen/Qwen2.5-3B-Instruct", "require_4bit": False, "min_tokens_per_sec": 5},
            {"name": "Qwen/Qwen2.5-1.5B-Instruct", "require_4bit": False, "min_tokens_per_sec": 0},
        ],
        "load_in_4bit": True, "max_new_tokens": 400,
        "temperature": 0.3, "top_p": 0.9, "context_length": 4096,
    },
    "translation": {
        "src_to_en_model": "ai4bharat/indictrans2-indic-en-dist-200M",
        "en_to_tgt_model": "ai4bharat/indictrans2-en-indic-dist-200M",
        "src_lang": "kan_Knda", "tgt_lang": "eng_Latn",
    },
    "hardware": {"assumed_gpu": "Tesla T4 (~15GB VRAM)"},
}

def set_seed(seed=SEED):
    random.seed(seed)
    try:
        import numpy as np; np.random.seed(seed)
    except Exception:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

set_seed()
print(f"[CONFIG] run_id={RUN_ID}  seed={SEED}")


[CONFIG] run_id=run_20260830_052920  seed=42


## SECTION 1b — Run-Mode Flags (PATCH 18)

First run should build the working core system only. Flip flags on once the core pipeline is verified.

In [2]:

# ============================================================
# SECTION 1b: RUN-MODE FLAGS — edit these, nothing else, to change scope
# ============================================================
RUN_FULL_CORPUS          = True    # False -> caps FarmerChat to DEBUG_FARMERCHAT_CAP records for a fast dry run
DEBUG_FARMERCHAT_CAP     = 5_000   # only used when RUN_FULL_CORPUS is False
RUN_EMBEDDING_ABLATION   = False   # question-only vs question+answer embedding comparison (Section 23b)
ENABLE_TRANSLATION       = True   # IndicTrans2 Kannada<->English (Section 14b)
RUN_EXPENSIVE_EVALUATION = True   # multi-call generation sweeps: top-K ablation, generation-eval table, latency sweep

CONFIG["translation"]["enabled"] = ENABLE_TRANSLATION

print(f"[RUN MODE] RUN_FULL_CORPUS={RUN_FULL_CORPUS} (cap={DEBUG_FARMERCHAT_CAP if not RUN_FULL_CORPUS else 'none'})")
print(f"[RUN MODE] RUN_EMBEDDING_ABLATION={RUN_EMBEDDING_ABLATION}")
print(f"[RUN MODE] ENABLE_TRANSLATION={ENABLE_TRANSLATION}")
print(f"[RUN MODE] RUN_EXPENSIVE_EVALUATION={RUN_EXPENSIVE_EVALUATION}")


[RUN MODE] RUN_FULL_CORPUS=True (cap=none)
[RUN MODE] RUN_EMBEDDING_ABLATION=False
[RUN MODE] ENABLE_TRANSLATION=True
[RUN MODE] RUN_EXPENSIVE_EVALUATION=True


## SECTION 2 — Environment / GPU Verification

In [3]:

# ============================================================
# SECTION 2: ENVIRONMENT / GPU DETECTION
# ============================================================
def detect_env():
    try:
        import google.colab  # noqa
        return "colab"
    except Exception:
        pass
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = detect_env()
print(f"[ENV] Detected environment: {ENV}")

def gpu_report():
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            print(f"[GPU] {name} | total VRAM ~ {total:.1f} GB")
            return True, total
        print("[GPU] CUDA not available -- using CPU (slower embeddings/generation).")
        return False, 0.0
    except Exception as e:
        print(f"[GPU] torch not importable yet ({e}).")
        return False, 0.0

GPU_AVAILABLE, GPU_VRAM_GB = gpu_report()


[ENV] Detected environment: colab
[GPU] Tesla T4 | total VRAM ~ 14.6 GB


## SECTION 3 — Dependencies / Version Verification

Run once per fresh runtime, then **restart the runtime**, then continue from the next section.

In [4]:

# ============================================================
# SECTION 3: DEPENDENCY INSTALLATION (pinned)
# ============================================================
PINNED = [
    "sentence-transformers==3.3.1",
    "faiss-cpu==1.9.0.post1",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "accelerate==1.1.1",
    "bitsandbytes==0.45.3",  # PATCH FINAL: 0.44.1 hits "No module named triton.ops" against
                              # the triton build shipped with recent Colab torch; 0.45.x fixed this.
    "sacrebleu==2.4.3",
    "sentencepiece==0.2.0",
    "IndicTransToolkit==1.1.1",
    "huggingface_hub==0.26.2",
    "ijson==3.3.0",     # PATCH 1/2: true streaming JSON-array parsing for FarmerChat
    "tqdm>=4.66",
]

def pip_install(pkgs):
    import subprocess
    for p in pkgs:
        print(f"[INSTALL] {p}")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  !! failed: {p}\n{r.stderr[-800:]}")

if os.environ.get("SKIP_INSTALL", "0") != "1":
    pip_install(PINNED)
    print("\n[INSTALL] Done. If this was a fresh runtime, RESTART now, then run Section 1 onward again.")
else:
    print("[INSTALL] Skipped (SKIP_INSTALL=1).")


[INSTALL] sentence-transformers==3.3.1
[INSTALL] faiss-cpu==1.9.0.post1
[INSTALL] transformers==4.46.3
[INSTALL] tokenizers==0.20.3
[INSTALL] accelerate==1.1.1
[INSTALL] bitsandbytes==0.45.3
[INSTALL] sacrebleu==2.4.3
[INSTALL] sentencepiece==0.2.0
[INSTALL] IndicTransToolkit==1.1.1
[INSTALL] huggingface_hub==0.26.2
[INSTALL] ijson==3.3.0
[INSTALL] tqdm>=4.66

[INSTALL] Done. If this was a fresh runtime, RESTART now, then run Section 1 onward again.


In [5]:

# ============================================================
# SECTION 3b: VERSION VERIFICATION (run AFTER restart)
# ============================================================
def verify_versions():
    import importlib
    checks = {"sentence_transformers": "3.3.1", "transformers": "4.46.3",
              "tokenizers": "0.20.3", "accelerate": "1.1.1", "faiss": None,
              "sacrebleu": "2.4.3", "ijson": "3.3.0"}
    report = {}
    for mod, expected in checks.items():
        try:
            m = importlib.import_module(mod)
            got = getattr(m, "__version__", "unknown")
            report[mod] = {"expected": expected, "got": got, "ok": (expected is None or got == expected)}
        except Exception as e:
            report[mod] = {"expected": expected, "got": None, "ok": False, "error": str(e)}
    for mod, r in report.items():
        print(f"[VERSION] {mod:<24} expected={r.get('expected')} got={r.get('got')}  "
              f"[{'OK' if r.get('ok') else 'MISMATCH/MISSING'}]")
    return report

VERSION_REPORT = verify_versions()


[VERSION] sentence_transformers    expected=3.3.1 got=3.3.1  [OK]
[VERSION] transformers             expected=4.46.3 got=4.46.3  [OK]
[VERSION] tokenizers               expected=0.20.3 got=0.20.3  [OK]
[VERSION] accelerate               expected=1.1.1 got=1.1.1  [OK]
[VERSION] faiss                    expected=None got=1.9.0  [OK]
[VERSION] sacrebleu                expected=2.4.3 got=2.4.3  [OK]
[VERSION] ijson                    expected=3.3.0 got=3.3.0  [OK]


## SECTION 4 — Google Drive Setup & Storage Layout

In [6]:

# ============================================================
# SECTION 4: STORAGE ROOT + DIRECTORY LAYOUT
# ============================================================
if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/majorprojectnlp")
elif ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working/majorprojectnlp")
else:
    PROJECT_ROOT = Path("./majorprojectnlp").resolve()

for d in ["data/raw", "data/processed", "data/final", "data/metadata",
          "embeddings", "indexes", "models", "evaluation", "logs", "configs", "runs"]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

CONFIG["paths"] = {
    "root": str(PROJECT_ROOT),
    "raw_dir": str(PROJECT_ROOT / "data/raw"),
    "processed_dir": str(PROJECT_ROOT / "data/processed"),
    "final_dir": str(PROJECT_ROOT / "data/final"),
    "metadata_dir": str(PROJECT_ROOT / "data/metadata"),
    "embeddings_dir": str(PROJECT_ROOT / "embeddings"),
    "indexes_dir": str(PROJECT_ROOT / "indexes"),
    "models_dir": str(PROJECT_ROOT / "models"),
    "evaluation_dir": str(PROJECT_ROOT / "evaluation"),
    "logs_dir": str(PROJECT_ROOT / "logs"),
    "configs_dir": str(PROJECT_ROOT / "configs"),
    "runs_dir": str(PROJECT_ROOT / "runs"),

    "dataset_26k": str(PROJECT_ROOT / "data" / CONFIG["datasets"]["dataset_26k_name"]),
    "farmerchat": str(PROJECT_ROOT / "data" / CONFIG["datasets"]["farmerchat_name"]),
    "farmerchat_shard_dir": str(PROJECT_ROOT / "data/raw" / "farmerchat"),

    "cleaned_26k": str(PROJECT_ROOT / "data/processed" / "cleaned_26k.jsonl"),
    "cleaned_farmerchat": str(PROJECT_ROOT / "data/processed" / "cleaned_farmerchat.jsonl"),
    "farmerchat_clean_progress": str(PROJECT_ROOT / "data/processed" / "farmerchat_clean_progress.json"),

    "final_kb": str(PROJECT_ROOT / "data/final" / "final_knowledge_base.jsonl"),
    "offset_index": str(PROJECT_ROOT / "data/metadata" / "offset_index.npy"),
    "provenance_stats": str(PROJECT_ROOT / "data/metadata" / "provenance_stats.json"),

    "embeddings_memmap": str(PROJECT_ROOT / "embeddings" / "kb_embeddings.f32"),
    "embeddings_meta": str(PROJECT_ROOT / "embeddings" / "embeddings_meta.json"),
    "embedding_progress": str(PROJECT_ROOT / "embeddings" / "progress.json"),

    "faiss_index": str(PROJECT_ROOT / "indexes" / "kb.index"),
    "faiss_config": str(PROJECT_ROOT / "indexes" / "faiss_config.json"),
    "faiss_build_progress": str(PROJECT_ROOT / "indexes" / "build_progress.json"),

    "run_log": str(PROJECT_ROOT / "logs" / f"{RUN_ID}.log"),
    "manifest": str(PROJECT_ROOT / "runs" / f"{RUN_ID}_manifest.json"),
    "eval_report": str(PROJECT_ROOT / "evaluation" / f"{RUN_ID}_eval_report.json"),
}
Path(CONFIG["paths"]["farmerchat_shard_dir"]).mkdir(parents=True, exist_ok=True)

logging.basicConfig(filename=CONFIG["paths"]["run_log"], level=logging.INFO,
                     format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("krishi_rag")

def log_step(name):
    def deco(fn):
        def wrapper(*a, **kw):
            t0 = time.time()
            logger.info(f"START {name}")
            print(f"\n=== {name} ===")
            try:
                out = fn(*a, **kw)
                dt = time.time() - t0
                logger.info(f"DONE {name} ({dt:.2f}s)")
                print(f"[OK] {name} finished in {dt:.2f}s")
                return out
            except Exception as e:
                logger.exception(f"FAILED {name}")
                print(f"[FAIL] {name}: {e}")
                raise
        return wrapper
    return deco

print(f"[STORAGE] PROJECT_ROOT = {PROJECT_ROOT}")
with open(str(PROJECT_ROOT / "configs" / f"{RUN_ID}_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)


Mounted at /content/drive
[STORAGE] PROJECT_ROOT = /content/drive/MyDrive/majorprojectnlp


## SECTION 5 — Hugging Face Authentication (optional)

All models used (BGE-M3, IndicTrans2, Qwen2.5) are public. This exists only for a future gated model. Token is never printed or hard-coded.

In [7]:

# ============================================================
# SECTION 5: OPTIONAL HUGGING FACE LOGIN
# ============================================================
ENABLE_HF_LOGIN = True

if ENABLE_HF_LOGIN:
    import getpass
    from huggingface_hub import login
    token = getpass.getpass("Enter Hugging Face token (input hidden): ")
    login(token=token, add_to_git_credential=False)
    del token
    print("[HF] Logged in (token not stored or printed).")
else:
    print("[HF] Skipped -- all models used are public.")


Enter Hugging Face token (input hidden): ··········
[HF] Logged in (token not stored or printed).


## SECTION 6 — True Streaming Reader + Dataset Schema Inspection (PATCH 1/2)

Never loads a whole multi-million-record file into memory. JSON arrays are streamed with `ijson` (does not materialize the array); JSONL/line files are iterated line by line straight from a file handle. Parse failures are reported, never silently dropped.

In [8]:

# ============================================================
# SECTION 6: STREAMING FILE READER (JSON array via ijson / JSONL via line iteration)
# ============================================================
import ijson

def _looks_like_array(fp):
    with open(fp, "rb") as f:
        head = f.read(4096).lstrip()
    return head.startswith(b"[")

def stream_raw_records(fp, fail_sink=None):
    '''Yields raw dict records from a single file without loading it whole into RAM.'''
    fp = str(fp)
    if _looks_like_array(fp):
        try:
            with open(fp, "rb") as f:
                for rec in ijson.items(f, "item"):
                    yield rec
            return
        except Exception as e:
            msg = f"{fp}: ijson streaming array parse failed ({e}); falling back to line-based parsing"
            if fail_sink is not None:
                fail_sink.append(msg)
            else:
                print("[LOADER]", msg)
    # JSONL / one-object-per-line fallback (also handles arrays with one bad top-level parse)
    with open(fp, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            line = line.strip().rstrip(",")
            if not line or line in ("[", "]"):
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                if fail_sink is not None:
                    fail_sink.append(f"{fp}:{i}: {e}")

def iter_raw_records(path):
    '''path may be a single file or a directory of .json/.jsonl shards. Streams
    everything; never returns a materialized list.'''
    path = Path(path)
    if not path.exists():
        print(f"[LOADER] MISSING: {path}")
        return
    if path.is_dir():
        shard_files = sorted(list(path.glob("*.jsonl")) + list(path.glob("*.json")))
        print(f"[LOADER] {path} is a directory with {len(shard_files)} shard(s).")
        for fp in shard_files:
            yield from stream_raw_records(fp)
    else:
        yield from stream_raw_records(path)

def inspect_schema(path, sample_n=2000):
    keys_seen, n, samples, fails = {}, 0, [], []
    gen = iter_raw_records(path) if not (isinstance(path, (str, Path)) and Path(path).is_dir()) \
        else iter_raw_records(path)
    for rec in gen:
        n += 1
        if isinstance(rec, dict):
            for k in rec.keys():
                keys_seen[k] = keys_seen.get(k, 0) + 1
        if len(samples) < 3:
            samples.append(rec)
        if n >= sample_n:
            break
    print(f"[SCHEMA] {path} -- inspected {n} record(s) (sample cap {sample_n})")
    print(f"[SCHEMA] Keys observed: {keys_seen}")
    for s in samples:
        print(f"[SCHEMA] sample: {json.dumps(s, ensure_ascii=False)[:300]}")
    return {"path": str(path), "n_inspected": n, "keys": keys_seen}

print("Inspecting dataset26k.json ...")
SCHEMA_26K = inspect_schema(CONFIG["paths"]["dataset_26k"])

print("\nResolving FarmerChat source (single file or shard directory) ...")
_fc_path = CONFIG["paths"]["farmerchat"]
if not Path(_fc_path).exists() and Path(CONFIG["paths"]["farmerchat_shard_dir"]).exists() \
        and any(Path(CONFIG["paths"]["farmerchat_shard_dir"]).iterdir()):
    _fc_path = CONFIG["paths"]["farmerchat_shard_dir"]
CONFIG["paths"]["farmerchat_resolved"] = _fc_path
print(f"[LOADER] FarmerChat resolved path: {_fc_path}")
SCHEMA_FARMERCHAT = inspect_schema(_fc_path, sample_n=2000)


Inspecting dataset26k.json ...
[SCHEMA] /content/drive/MyDrive/majorprojectnlp/data/dataset26k.json -- inspected 2000 record(s) (sample cap 2000)
[SCHEMA] Keys observed: {'question': 2000, 'answers': 2000}
[SCHEMA] sample: {"question": "why is crop rotation important in farming?", "answers": "This helps to prevent soil erosion and depletion, and can also help to control pests and diseases"}
[SCHEMA] sample: {"question": "What farming practice helps prevent soil erosion?", "answers": "Crop Rotation"}
[SCHEMA] sample: {"question": "what is crop rotation", "answers": "Crop rotation is the practice of growing a series of different crops in the same area over several seasons"}

Resolving FarmerChat source (single file or shard directory) ...
[LOADER] FarmerChat resolved path: /content/drive/MyDrive/majorprojectnlp/data/farmerchat.json
[SCHEMA] /content/drive/MyDrive/majorprojectnlp/data/farmerchat.json -- inspected 2000 record(s) (sample cap 2000)
[SCHEMA] Keys observed: {'asset_type': 2000

## SECTION 7 — Shared Cleaning Utilities

In [9]:

# ============================================================
# SECTION 7: FIELD NORMALIZATION + CLEANING HELPERS (shared by both sources)
# ============================================================
QUESTION_KEYS = ["question", "query", "instruction", "prompt", "q"]
ANSWER_KEYS = ["answer", "answers", "response", "output", "a"]

BOILERPLATE_PATTERNS = [
    re.compile(r"^\s*(n/?a|none|null|todo|test|sample)\s*$", re.IGNORECASE),
    re.compile(r"^\s*<.*?>\s*$"),
    re.compile(r"^\s*\.{3,}\s*$"),
]

def normalize_text(s):
    if s is None:
        return ""
    if isinstance(s, list):
        s = " ".join(str(x) for x in s if x)
    s = unicodedata.normalize("NFC", str(s))
    s = s.replace("\u200b", "").replace("\ufeff", "")
    return re.sub(r"\s+", " ", s).strip()

def is_boilerplate(s):
    if not s:
        return True
    return any(pat.match(s) for pat in BOILERPLATE_PATTERNS)

def extract_qa(rec):
    if not isinstance(rec, dict):
        return None, None
    q = next((rec[k] for k in QUESTION_KEYS if rec.get(k)), None)
    a = next((rec[k] for k in ANSWER_KEYS if rec.get(k)), None)
    return q, a

def compact_hash(question, answer):
    '''16-byte blake2b digest as a Python int -- far cheaper to hold in a
    dedup set than a 64-char sha256 hex string, at multi-million scale.'''
    h = hashlib.blake2b((question.lower() + "||" + answer.lower()).encode("utf-8"), digest_size=16)
    return int.from_bytes(h.digest(), "big")

def clean_and_validate_record(rec):
    '''raw dict -> (status, cleaned_record_or_None). status in
    {'ok','malformed','missing_q','missing_a','boilerplate'}.'''
    if not isinstance(rec, dict):
        return "malformed", None
    q_raw, a_raw = extract_qa(rec)
    q, a = normalize_text(q_raw), normalize_text(a_raw)
    if not q:
        return "missing_q", None
    if not a:
        return "missing_a", None
    if is_boilerplate(q) or is_boilerplate(a):
        return "boilerplate", None
    provenance = {k: rec.get(k) for k in
                  ["id", "source", "category", "crop", "state", "district",
                   "verified", "author", "date", "language"] if k in rec}
    return "ok", {"question": q, "answer": a, "record_hash": compact_hash(q, a), "provenance": provenance}


## SECTION 8 — 26K Preprocessing (small dataset, in-memory is fine)

In [10]:

# ============================================================
# SECTION 8: 26K PREPROCESSING (small corpus -> in-memory list is safe)
# ============================================================
def clean_26k(path, source_name="dataset_26k"):
    stats = {"source": source_name, "raw": 0, "malformed": 0, "missing_q": 0,
              "missing_a": 0, "boilerplate": 0, "clean": 0, "within_source_duplicates": 0}
    audit = {"missing_q": [], "missing_a": [], "malformed": [], "boilerplate": []}
    seen, out = set(), []
    for rec in iter_raw_records(path):
        stats["raw"] += 1
        status, clean = clean_and_validate_record(rec)
        if status != "ok":
            stats[status] += 1
            if len(audit.get(status, [])) < 200:
                audit.setdefault(status, []).append(json.dumps(rec, ensure_ascii=False)[:200])
            continue
        if clean["record_hash"] in seen:
            stats["within_source_duplicates"] += 1
            continue
        seen.add(clean["record_hash"])
        clean["source_dataset"] = source_name
        out.append(clean)
    stats["clean"] = len(out)
    return out, stats, audit

CLEAN_26K, STATS_26K, AUDIT_26K = clean_26k(CONFIG["paths"]["dataset_26k"])
with open(CONFIG["paths"]["cleaned_26k"], "w", encoding="utf-8") as f:
    for r in CLEAN_26K:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"[26K] stats: {STATS_26K}")
for k, v in AUDIT_26K.items():
    if v:
        print(f"[26K][audit:{k}] e.g. {v[:3]}")


[26K] stats: {'source': 'dataset_26k', 'raw': 26088, 'malformed': 0, 'missing_q': 0, 'missing_a': 0, 'boilerplate': 0, 'clean': 5804, 'within_source_duplicates': 20284}


## SECTION 9 — FarmerChat Preprocessing (streaming, disk-backed, resumable) — PATCH 1

No multi-million-item Python list is ever created. Each valid unique record is written immediately to disk. For a directory of shards, already-completed shards are skipped on restart. For a single huge file, restarting reloads the hashes already written and simply re-streams the source (cheap, CPU-bound text work) — records already on disk are recognized as duplicates and skipped, so nothing is lost or duplicated.

In [11]:

# ============================================================
# SECTION 9: FARMERCHAT STREAMING CLEAN -> DISK (memory-bounded, restart-safe)
# ============================================================
def _load_progress(path, default):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default

def _save_progress(path, progress):
    with open(path, "w") as f:
        json.dump(progress, f, indent=2, default=str)

def _reload_seen_hashes_from_output(output_path):
    seen = set()
    n = 0
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    seen.add(rec["record_hash"])
                    n += 1
                except Exception:
                    continue
    if n:
        print(f"[FarmerChat][resume] reloaded {n:,} existing clean records / hashes from {output_path}")
    return seen

@log_step("FarmerChat streaming preprocessing (disk-backed)")
def clean_farmerchat_streaming(source_path, output_path, progress_path,
                                source_name="farmerchat", report_every=100_000, cap=None):
    source_path = Path(source_path)
    is_dir = source_path.is_dir()
    default_progress = {"completed_shards": [], "stats": {
        "source": source_name, "raw": 0, "malformed": 0, "missing_q": 0,
        "missing_a": 0, "boilerplate": 0, "clean": 0, "within_source_duplicates": 0}}
    progress = _load_progress(progress_path, default_progress)
    stats = progress["stats"]
    completed_shards = set(progress.get("completed_shards", []))

    seen_hashes = _reload_seen_hashes_from_output(output_path)
    audit = {"missing_q": [], "missing_a": [], "malformed": [], "boilerplate": []}

    def process_stream(records_iter, shard_label=None):
        nonlocal stats
        written_here = 0
        with open(output_path, "a", encoding="utf-8") as out_f:
            for rec in records_iter:
                stats["raw"] += 1
                if stats["raw"] % report_every == 0:
                    print(f"[FarmerChat] processed={stats['raw']:,} valid={stats['clean']:,} "
                          f"invalid={stats['malformed']+stats['missing_q']+stats['missing_a']+stats['boilerplate']:,} "
                          f"duplicates={stats['within_source_duplicates']:,} written={stats['clean']:,}"
                          + (f" [shard={shard_label}]" if shard_label else ""))
                    _save_progress(progress_path, progress)
                status, clean = clean_and_validate_record(rec)
                if status != "ok":
                    stats[status] += 1
                    if len(audit.get(status, [])) < 200:
                        audit.setdefault(status, []).append(json.dumps(rec, ensure_ascii=False)[:200])
                    continue
                if clean["record_hash"] in seen_hashes:
                    stats["within_source_duplicates"] += 1
                    continue
                seen_hashes.add(clean["record_hash"])
                clean["source_dataset"] = source_name
                out_f.write(json.dumps(clean, ensure_ascii=False) + "\n")
                stats["clean"] += 1
                written_here += 1
                if cap is not None and stats["clean"] >= cap:
                    print(f"[FarmerChat] DEBUG cap of {cap:,} clean records reached -- stopping early "
                          f"(RUN_FULL_CORPUS=False).")
                    return True  # signal: stop everything
        return False

    stop = False
    if is_dir:
        shard_files = sorted(list(source_path.glob("*.jsonl")) + list(source_path.glob("*.json")))
        print(f"[FarmerChat] {len(shard_files)} shard(s) found; {len(completed_shards)} already complete.")
        for fp in shard_files:
            if str(fp) in completed_shards:
                continue
            stop = process_stream(stream_raw_records(fp), shard_label=fp.name)
            if not stop:
                completed_shards.add(str(fp))
                progress["completed_shards"] = sorted(completed_shards)
                _save_progress(progress_path, progress)
            if stop:
                break
    else:
        stop = process_stream(stream_raw_records(source_path))

    stats["clean"] = stats["clean"]
    progress["stats"] = stats
    _save_progress(progress_path, progress)
    print(f"[FarmerChat] FINAL: {stats}")
    return stats, audit

_cap = None if RUN_FULL_CORPUS else DEBUG_FARMERCHAT_CAP
STATS_FARMERCHAT, AUDIT_FARMERCHAT = clean_farmerchat_streaming(
    CONFIG["paths"]["farmerchat_resolved"], CONFIG["paths"]["cleaned_farmerchat"],
    CONFIG["paths"]["farmerchat_clean_progress"], cap=_cap,
)
for k, v in AUDIT_FARMERCHAT.items():
    if v:
        print(f"[FarmerChat][audit:{k}] e.g. {v[:3]}")



=== FarmerChat streaming preprocessing (disk-backed) ===
[FarmerChat] processed=100,000 valid=84,971 invalid=0 duplicates=15,028 written=84,971
[FarmerChat] processed=200,000 valid=154,903 invalid=6 duplicates=45,090 written=154,903
[FarmerChat] FINAL: {'source': 'farmerchat', 'raw': 238030, 'malformed': 0, 'missing_q': 0, 'missing_a': 12, 'boilerplate': 0, 'clean': 182348, 'within_source_duplicates': 55670}
[OK] FarmerChat streaming preprocessing (disk-backed) finished in 59.29s
[FarmerChat][audit:missing_a] e.g. ['{"asset_type": "crop", "asset_name": "Chili", "query": "How can I identify the presence of thrips in my chili plants?", "response": null, "user_country": "India", "user_geo_level2": "Andhra Pradesh", ', '{"asset_type": "crop", "asset_name": "generic", "query": "ಹಣ್ಣು ಹಕ್ಕುಗಳಿಗೆ ಹಾನಿಕಾರಕ ಕೀಟಗಳ ವಿರುದ್ಧ ಯಾವ ರಾಸಾಯನಿಕವನ್ನು ಬಳಸಬೇಕು?", "response": null, "user_country": "India", "user_geo_level2": "Karnataka"', '{"asset_type": "crop", "asset_name": "Rice", "query": "What fungicide

## SECTION 10 — Streaming Cross-Source Dedup + Final KB Construction (offset-indexed) — PATCH 4

Reads the small 26K list (in memory) and streams the on-disk FarmerChat-cleaned file line by line. Writes the deduped final knowledge base to disk and simultaneously builds a compact `int64` NumPy array mapping `record_id -> byte offset` — this **replaces** the old `{id: {...}}` metadata dict entirely. No full corpus is ever held in RAM.

In [12]:

# ============================================================
# SECTION 10: MERGE + CROSS-SOURCE DEDUP + FINAL KB (streaming, offset-indexed)
# ============================================================
import numpy as np

@log_step("Streaming merge + cross-source dedup + final KB construction")
def build_final_kb_streaming(clean_26k_records, cleaned_farmerchat_path, final_kb_path):
    seen = set()
    offsets = []
    cross_dupes = 0
    source_counts = {"dataset_26k": 0, "farmerchat": 0}

    def emit_stream():
        for r in clean_26k_records:
            yield r
        if os.path.exists(cleaned_farmerchat_path):
            with open(cleaned_farmerchat_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        yield json.loads(line)
                    except Exception:
                        continue

    with open(final_kb_path, "w", encoding="utf-8") as out_f:
        for rec in emit_stream():
            h = rec["record_hash"]
            if h in seen:
                cross_dupes += 1
                continue
            seen.add(h)
            record_id = len(offsets)
            final_rec = {
                "record_id": record_id,
                "question": rec["question"],
                "answer": rec["answer"],
                "source_dataset": rec["source_dataset"],
                "provenance": rec.get("provenance", {}),
            }
            offsets.append(out_f.tell())
            out_f.write(json.dumps(final_rec, ensure_ascii=False) + "\n")
            source_counts[rec["source_dataset"]] = source_counts.get(rec["source_dataset"], 0) + 1

    offset_arr = np.array(offsets, dtype=np.int64)
    np.save(CONFIG["paths"]["offset_index"], offset_arr)
    return {
        "cross_source_duplicates_removed": cross_dupes,
        "final_count": len(offsets),
        "source_breakdown": source_counts,
    }

MERGE_STATS = build_final_kb_streaming(CLEAN_26K, CONFIG["paths"]["cleaned_farmerchat"], CONFIG["paths"]["final_kb"])
print(f"[MERGE] {MERGE_STATS}")

if MERGE_STATS["final_count"] == 0:
    raise RuntimeError("[FATAL] Final knowledge base is empty after cleaning -- check dataset paths/schema.")

# free the only remaining full-corpus-scale in-memory structure (26K is small, but be tidy)
del CLEAN_26K
gc.collect()



=== Streaming merge + cross-source dedup + final KB construction ===
[OK] Streaming merge + cross-source dedup + final KB construction finished in 38.30s
[MERGE] {'cross_source_duplicates_removed': 0, 'final_count': 188152, 'source_breakdown': {'dataset_26k': 5804, 'farmerchat': 182348}}


213

## SECTION 11 — Access Layer: Offset-Indexed Record Store (replaces `KB_RECORDS`)

In [13]:

# ============================================================
# SECTION 11: RECORD ACCESS BY ID -- O(1) SEEK, NO FULL-CORPUS LIST IN RAM
# ============================================================
OFFSET_INDEX = np.load(CONFIG["paths"]["offset_index"])
N_TOTAL_RECORDS = len(OFFSET_INDEX)
print(f"[KB] N_TOTAL_RECORDS = {N_TOTAL_RECORDS:,} (offset index loaded, {OFFSET_INDEX.nbytes/1e6:.2f} MB)")

_kb_file_handle = open(CONFIG["paths"]["final_kb"], "rb")

def get_record(record_id):
    if record_id < 0 or record_id >= N_TOTAL_RECORDS:
        raise IndexError(f"record_id {record_id} out of range [0, {N_TOTAL_RECORDS})")
    _kb_file_handle.seek(int(OFFSET_INDEX[record_id]))
    line = _kb_file_handle.readline()
    return json.loads(line)

def iter_kb_records(start=0, end=None):
    '''Streams records [start, end) in order without holding them all in RAM.'''
    end = N_TOTAL_RECORDS if end is None else min(end, N_TOTAL_RECORDS)
    with open(CONFIG["paths"]["final_kb"], "rb") as f:
        f.seek(int(OFFSET_INDEX[start]))
        for _ in range(start, end):
            yield json.loads(f.readline())

def sample_record_ids(n, seed=SEED):
    rng = random.Random(seed)
    n = min(n, N_TOTAL_RECORDS)
    return rng.sample(range(N_TOTAL_RECORDS), n)

def sample_records(n, seed=SEED):
    return [get_record(i) for i in sample_record_ids(n, seed=seed)]

# sanity check
_r0 = get_record(0)
assert "question" in _r0 and "answer" in _r0, "Access-layer sanity check failed."
print(f"[KB] sample record[0]: {json.dumps(_r0, ensure_ascii=False)[:250]}")


[KB] N_TOTAL_RECORDS = 188,152 (offset index loaded, 1.51 MB)
[KB] sample record[0]: {"record_id": 0, "question": "why is crop rotation important in farming?", "answer": "This helps to prevent soil erosion and depletion, and can also help to control pests and diseases", "source_dataset": "dataset_26k", "provenance": {}}


## SECTION 12 — Final Knowledge-Base Provenance Statistics

In [14]:

# ============================================================
# SECTION 12: PROVENANCE STATISTICS
# ============================================================
PROVENANCE_STATS = {
    "run_id": RUN_ID, "dataset_26k": STATS_26K, "farmerchat": STATS_FARMERCHAT,
    "cross_source_duplicates_removed": MERGE_STATS["cross_source_duplicates_removed"],
    "final_count": MERGE_STATS["final_count"], "source_breakdown": MERGE_STATS["source_breakdown"],
    "run_mode": {"full_corpus": RUN_FULL_CORPUS, "debug_cap": None if RUN_FULL_CORPUS else DEBUG_FARMERCHAT_CAP},
}
with open(CONFIG["paths"]["provenance_stats"], "w") as f:
    json.dump(PROVENANCE_STATS, f, indent=2, default=str)
print(json.dumps(PROVENANCE_STATS, indent=2, default=str))


{
  "run_id": "run_20260830_052920",
  "dataset_26k": {
    "source": "dataset_26k",
    "raw": 26088,
    "malformed": 0,
    "missing_q": 0,
    "missing_a": 0,
    "boilerplate": 0,
    "clean": 5804,
    "within_source_duplicates": 20284
  },
  "farmerchat": {
    "source": "farmerchat",
    "raw": 238030,
    "malformed": 0,
    "missing_q": 0,
    "missing_a": 12,
    "boilerplate": 0,
    "clean": 182348,
    "within_source_duplicates": 55670
  },
  "cross_source_duplicates_removed": 0,
  "final_count": 188152,
  "source_breakdown": {
    "dataset_26k": 5804,
    "farmerchat": 182348
  },
  "run_mode": {
    "full_corpus": true,
    "debug_cap": null
  }
}


## SECTION 13 — Embedding Model Loading (BGE-M3)

In [15]:

# ============================================================
# SECTION 13: LOAD BGE-M3
# ============================================================
import torch
from sentence_transformers import SentenceTransformer

GPU_AVAILABLE, GPU_VRAM_GB = gpu_report()
DEVICE = "cuda" if GPU_AVAILABLE else "cpu"

@log_step("Load BGE-M3 embedding model")
def load_embedding_model():
    model = SentenceTransformer(CONFIG["embedding"]["model_name"], device=DEVICE)
    model.max_seq_length = CONFIG["embedding"]["max_seq_length"]
    return model

embed_model = load_embedding_model()
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"[EMBED] model={CONFIG['embedding']['model_name']} dim={EMBED_DIM} device={DEVICE} "
      f"normalize={CONFIG['embedding']['normalize']} batch_size={CONFIG['embedding']['batch_size']} "
      f"max_seq_length={CONFIG['embedding']['max_seq_length']}")


[GPU] Tesla T4 | total VRAM ~ 14.6 GB

=== Load BGE-M3 embedding model ===


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[OK] Load BGE-M3 embedding model finished in 35.06s
[EMBED] model=BAAI/bge-m3 dim=1024 device=cuda normalize=True batch_size=64 max_seq_length=512


## SECTION 14 — Local LLM Loading (loaded once, smoke-tested) — PATCH 8/14

Moved earlier so the small-scale end-to-end smoke test (Section 15) can exercise the real generator before hours of full-corpus embedding begin.

In [16]:
import torch
import gc

# ============================================================
# SECTION 14: LOAD LOCAL INSTRUCTION LLM
# PATCH FINAL (items 9-11): the previous version caught any 4-bit load failure
# and silently fell back to full-precision Qwen2.5-7B, which is unusably slow
# on a T4 (multi-minute generations). This version:
#   1. Runs a REAL bitsandbytes capability smoke test (not just "did it import")
#      before ever attempting a 4-bit load.
#   2. Treats "require_4bit" per-candidate: the 7B model is ONLY ever loaded
#      4-bit -- if that is not reliably possible, it is skipped entirely rather
#      than loaded unquantized.
#   3. Measures tokens/sec on a real smoke generation and rejects a candidate
#      that is too slow, moving to the next (smaller) candidate instead.
#   4. The final candidate in CONFIG["generation"]["candidate_models"] has
#      min_tokens_per_sec=0, so the chain always terminates in a usable model.
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def verify_bnb_quantization():
    """Real capability check: actually run a tiny 4-bit matmul on this GPU,
    rather than trusting that `import bitsandbytes` succeeding means 4-bit works.
    This is what catches the 'No module named triton.ops' class of failure,
    which otherwise only surfaces mid-way through loading a multi-GB model."""
    if not GPU_AVAILABLE:
        return False, "no GPU available"
    try:
        import bitsandbytes as bnb
        cc_major, _ = torch.cuda.get_device_capability(0)
        if cc_major < 7:
            return False, f"GPU compute capability {cc_major}.x too old for reliable 4-bit nf4"
        probe = torch.nn.Linear(64, 64)
        q_probe = bnb.nn.Linear4bit(64, 64, compute_dtype=torch.float16)
        q_probe.load_state_dict(probe.state_dict(), strict=False)
        q_probe = q_probe.to("cuda")
        x = torch.randn(2, 64, dtype=torch.float16, device="cuda")
        with torch.no_grad():
            _ = q_probe(x)
        del probe, q_probe, x
        gc.collect(); torch.cuda.empty_cache()
        return True, f"bitsandbytes {getattr(bnb, '__version__', '?')} 4-bit smoke test passed"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


BNB_4BIT_OK, BNB_4BIT_MSG = verify_bnb_quantization()
print(f"[BNB] 4-bit quantization available: {BNB_4BIT_OK} ({BNB_4BIT_MSG})")


def unload_model(tok, model):
    try:
        del model, tok
    except Exception:
        pass
    gc.collect()
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()


def load_causal_lm(name, use_4bit):
    tok = AutoTokenizer.from_pretrained(name)
    bnb_config = None
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(
        name, quantization_config=bnb_config,
        device_map="auto" if GPU_AVAILABLE else None,
        torch_dtype=torch.float16 if GPU_AVAILABLE else torch.float32,
    )
    if not GPU_AVAILABLE:
        model = model.to("cpu")
    return tok, model


def smoke_test(tok, model):
    msgs = [{"role": "user", "content": "Say 'ready' if you can read this."}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    gen_s = time.time() - t0
    n_new = out.shape[1] - inputs["input_ids"].shape[1]
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    tok_per_s = n_new / gen_s if gen_s > 0 else 0.0
    return gen_s, n_new, tok_per_s, text


@log_step("Local LLM load + capability-gated selection")
def load_generation_model():
    candidates = CONFIG["generation"]["candidate_models"]
    last_err = None
    for spec in candidates:
        name, require_4bit, min_tps = spec["name"], spec["require_4bit"], spec["min_tokens_per_sec"]
        is_last_candidate = (spec is candidates[-1])

        if require_4bit and not BNB_4BIT_OK:
            print(f"[LLM] Skipping {name}: requires 4-bit, but 4-bit is not reliably available "
                  f"on this runtime ({BNB_4BIT_MSG}).")
            continue

        attempt_4bit = BNB_4BIT_OK and CONFIG["generation"]["load_in_4bit"]
        try:
            print(f"[LLM] Attempting {name} ({'4-bit' if attempt_4bit else 'fp16/fp32'})...")
            t0 = time.time()
            tok, model = load_causal_lm(name, use_4bit=attempt_4bit)
            load_s = time.time() - t0
        except Exception as e_first:
            if require_4bit:
                # No unquantized fallback for a model that REQUIRES 4-bit -- move on.
                print(f"[LLM] {name} 4-bit load failed ({e_first}); this model requires 4-bit, "
                      f"so it is skipped rather than loaded unquantized.")
                last_err = e_first
                continue
            # Small/medium models are allowed a single fp16 retry.
            try:
                print(f"[LLM] {name} 4-bit load failed ({e_first}); retrying in fp16/fp32 "
                      f"(model is small enough to be safe unquantized on T4).")
                gc.collect()
                if GPU_AVAILABLE:
                    torch.cuda.empty_cache()
                t0 = time.time()
                tok, model = load_causal_lm(name, use_4bit=False)
                load_s = time.time() - t0
            except Exception as e_second:
                print(f"[LLM] {name} failed in both 4-bit and fp16/fp32: {e_second}")
                last_err = e_second
                continue

        try:
            gen_s, n_new, tok_per_s, sample_text = smoke_test(tok, model)
        except Exception as e_gen:
            print(f"[LLM] {name} loaded but smoke generation failed: {e_gen}")
            unload_model(tok, model)
            last_err = e_gen
            continue

        quant = "4-bit nf4" if (hasattr(model, "is_loaded_in_4bit") and model.is_loaded_in_4bit) else "fp16/fp32"
        vram_gb = torch.cuda.memory_allocated() / (1024**3) if GPU_AVAILABLE else 0.0
        n_params = sum(p.numel() for p in model.parameters())
        print(f"[LLM] {name} | params={n_params/1e9:.2f}B | quant={quant} | load={load_s:.1f}s | "
              f"smoke_gen={gen_s:.2f}s for {n_new} tok ({tok_per_s:.1f} tok/s) | "
              f"vram~{vram_gb:.2f}GB | reply='{sample_text.strip()}'")

        if tok_per_s < min_tps and not is_last_candidate:
            print(f"[LLM] {name} generation speed {tok_per_s:.1f} tok/s is below the "
                  f"{min_tps} tok/s practicality threshold -- rejecting and trying a smaller model.")
            unload_model(tok, model)
            last_err = RuntimeError(f"{name} too slow: {tok_per_s:.1f} tok/s < {min_tps}")
            continue

        report = {
            "model": name, "parameters_B": round(n_params / 1e9, 2), "quantization": quant,
            "vram_gb": round(vram_gb, 2), "context_length": CONFIG["generation"]["context_length"],
            "load_time_s": round(load_s, 1), "smoke_generation_latency_s": round(gen_s, 2),
            "smoke_tokens_per_sec": round(tok_per_s, 1),
        }
        print(f"[LLM] SELECTED: {name} ({quant}) -- {tok_per_s:.1f} tok/s meets practicality bar.")
        return tok, model, report

    raise RuntimeError(f"All candidate LLMs failed to load or were rejected as impractical. "
                        f"Last error: {last_err}")


LLM_TOKENIZER, LLM_MODEL, LLM_REPORT = load_generation_model()
print(f"[LLM] Report: {json.dumps(LLM_REPORT, indent=2)}")


[BNB] 4-bit quantization available: True (bitsandbytes 0.45.3 4-bit smoke test passed)

=== Local LLM load + capability-gated selection ===
[LLM] Attempting Qwen/Qwen2.5-7B-Instruct (4-bit)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[LLM] Qwen/Qwen2.5-3B-Instruct | params=1.70B | quant=4-bit nf4 | load=142.2s | smoke_gen=0.24s for 2 tok (8.2 tok/s) | vram~9.26GB | reply='ready'
[LLM] SELECTED: Qwen/Qwen2.5-3B-Instruct (4-bit nf4) -- 8.2 tok/s meets practicality bar.
[OK] Local LLM load + capability-gated selection finished in 571.46s
[LLM] Report: {
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "parameters_B": 1.7,
  "quantization": "4-bit nf4",
  "vram_gb": 9.26,
  "context_length": 4096,
  "load_time_s": 142.2,
  "smoke_generation_latency_s": 0.24,
  "smoke_tokens_per_sec": 8.2
}


## SECTION 14b — Optional Translation (IndicTrans2) — gated by `ENABLE_TRANSLATION`

In [17]:

# ============================================================
# SECTION 14b: INDICTRANS2 (bidirectional), validated fix preserved, OFF by default
# ============================================================
TRANSLATION_READY = True
if CONFIG["translation"]["enabled"]:
    try:
        from IndicTransToolkit.processor import IndicProcessor
        # PATCH FINAL: IndicTrans2 is an encoder-decoder (seq2seq) model, not causal LM.
        # Loading it with AutoModelForCausalLM either fails or silently mis-generates.
        from transformers import AutoModelForSeq2SeqLM

        @log_step("Load IndicTrans2 (both directions)")
        def load_translation_models():
            ip = IndicProcessor(inference=True)
            s2e_name = CONFIG["translation"]["src_to_en_model"]
            e2t_name = CONFIG["translation"]["en_to_tgt_model"]
            s2e_tok = AutoTokenizer.from_pretrained(s2e_name, trust_remote_code=True)
            s2e_model = AutoModelForSeq2SeqLM.from_pretrained(
                s2e_name, trust_remote_code=True,
                torch_dtype=torch.float16 if GPU_AVAILABLE else torch.float32).to(DEVICE if GPU_AVAILABLE else "cpu")
            e2t_tok = AutoTokenizer.from_pretrained(e2t_name, trust_remote_code=True)
            e2t_model = AutoModelForSeq2SeqLM.from_pretrained(
                e2t_name, trust_remote_code=True,
                torch_dtype=torch.float16 if GPU_AVAILABLE else torch.float32).to(DEVICE if GPU_AVAILABLE else "cpu")
            return ip, (s2e_tok, s2e_model), (e2t_tok, e2t_model)

        IP, (SRC2EN_TOK, SRC2EN_MODEL), (EN2TGT_TOK, EN2TGT_MODEL) = load_translation_models()
        TRANSLATION_READY = True

        def translate(text, direction):
            if direction == "src2en":
                tok, model = SRC2EN_TOK, SRC2EN_MODEL
                batch = IP.preprocess_batch([text], src_lang=CONFIG["translation"]["src_lang"],
                                             tgt_lang=CONFIG["translation"]["tgt_lang"])
            else:
                tok, model = EN2TGT_TOK, EN2TGT_MODEL
                batch = IP.preprocess_batch([text], src_lang=CONFIG["translation"]["tgt_lang"],
                                             tgt_lang=CONFIG["translation"]["src_lang"])
            inputs = tok(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=256, num_beams=5)
            decoded = tok.batch_decode(out, skip_special_tokens=True)
            tgt = CONFIG["translation"]["tgt_lang"] if direction == "src2en" else CONFIG["translation"]["src_lang"]
            return IP.postprocess_batch(decoded, lang=tgt)[0]

        print("[TRANSLATION] IndicTrans2 loaded (both directions).")
    except Exception as e:
        print(f"[TRANSLATION] Disabled -- could not load IndicTrans2: {e}")
        TRANSLATION_READY = False
else:
    print("[TRANSLATION] Disabled by ENABLE_TRANSLATION flag.")



=== Load IndicTrans2 (both directions) ===


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

[OK] Load IndicTrans2 (both directions) finished in 50.02s
[TRANSLATION] IndicTrans2 loaded (both directions).


## SECTION 15 — Small-Scale Smoke Test (PATCH 19) — must pass before full-corpus embedding

In [18]:
import numpy as np
import faiss as _faiss_smoke

@log_step("Small-scale smoke test (~300 records)")
def run_smoke_test(n=300):

    sample = sample_records(n)

    assert len(sample) > 0, "Smoke test: no records available."

    # ------------------------------------------------------------
    # 1. Embedding
    # ------------------------------------------------------------
    texts = [r["question"] for r in sample]

    vecs = embed_model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    vecs = np.asarray(vecs, dtype=np.float32)

    assert vecs.shape[0] == len(sample), \
        "Smoke test: embedding count mismatch."

    assert np.isfinite(vecs).all(), \
        "Smoke test: embeddings contain NaN/Inf."

    # ------------------------------------------------------------
    # 2. FAISS
    # ------------------------------------------------------------
    mini_index = _faiss_smoke.IndexFlatIP(vecs.shape[1])

    mini_index.add(
        np.ascontiguousarray(vecs, dtype=np.float32)
    )

    assert mini_index.ntotal == len(sample), \
        "Smoke test: FAISS insertion count mismatch."

    # ------------------------------------------------------------
    # 3. Retrieval
    # ------------------------------------------------------------
    q_vec = embed_model.encode(
        [sample[0]["question"]],
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    q_vec = np.asarray(q_vec, dtype=np.float32)

    D, I = mini_index.search(
        np.ascontiguousarray(q_vec, dtype=np.float32),
        min(5, len(sample)),
    )

    retrieved_ids = I[0].tolist()
    retrieved_scores = D[0].tolist()

    # IMPORTANT:
    # Do not require record 0 to be rank #1.
    # Another record may have the same/nearly identical question.
    assert 0 in retrieved_ids, (
        "Smoke test: query's own vector was not found in the Top-K results. "
        f"Retrieved IDs={retrieved_ids}, scores={retrieved_scores}"
    )

    self_rank = retrieved_ids.index(0) + 1
    self_score = retrieved_scores[retrieved_ids.index(0)]

    # For normalized vectors, exact self-similarity should be approximately 1.
    assert self_score > 0.99, (
        f"Smoke test: self-retrieval similarity unexpectedly low: {self_score:.4f}"
    )

    top_record = sample[retrieved_ids[0]]

    assert "question" in top_record and "answer" in top_record, \
        "Smoke test: QA record reconstruction failed."

    # ------------------------------------------------------------
    # 4. LLM generation
    # ------------------------------------------------------------
    mini_context = (
        f"[RECORD 1]\n"
        f"Question: {top_record['question']}\n"
        f"Answer: {top_record['answer']}"
    )

    msgs = [{
        "role": "user",
        "content": (
            "Using only the following agricultural information, "
            "answer the farmer's question briefly and accurately.\n\n"
            f"{mini_context}\n\n"
            f"Question: {sample[0]['question']}"
        )
    }]

    prompt = LLM_TOKENIZER.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = LLM_TOKENIZER(
        prompt,
        return_tensors="pt"
    ).to(LLM_MODEL.device)

    out = LLM_MODEL.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
    )

    gen_text = LLM_TOKENIZER.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    assert len(gen_text.strip()) > 0, \
        "Smoke test: LLM produced an empty response."

    # ------------------------------------------------------------
    # 5. Final report
    # ------------------------------------------------------------
    print(
        f"[SMOKE] embedding OK ({vecs.shape}) | "
        f"FAISS OK ({mini_index.ntotal} vectors) | "
        f"retrieval OK | "
        f"self-rank={self_rank} | "
        f"self-score={self_score:.4f} | "
        f"QA reconstruction OK | "
        f"LLM generation OK: '{gen_text.strip()[:120]}'"
    )

    return True


SMOKE_TEST_PASSED = run_smoke_test(
    n=min(300, N_TOTAL_RECORDS)
)

if not SMOKE_TEST_PASSED:
    raise RuntimeError(
        "Smoke test failed -- fix before running full-corpus embedding."
    )

print("[SMOKE] PASSED. Proceeding to full-corpus embedding.")


=== Small-scale smoke test (~300 records) ===
[SMOKE] embedding OK ((300, 1024)) | FAISS OK (300 vectors) | retrieval OK | self-rank=2 | self-score=1.0000 | QA reconstruction OK | LLM generation OK: 'Based on the agricultural information provided, you should irrigate wheat during the tillering stage in [REDACTED], [RED'
[OK] Small-scale smoke test (~300 records) finished in 6.70s
[SMOKE] PASSED. Proceeding to full-corpus embedding.


## SECTION 16 — Embedding Storage / Resume System

In [19]:

# ============================================================
# SECTION 16: RESUMABLE EMBEDDING CHECKPOINTING (disk-backed memmap)
# ============================================================
def estimate_storage(n_docs, dim):
    emb_bytes = n_docs * dim * 4
    meta_bytes = n_docs * 250
    print(f"[ESTIMATE] documents={n_docs:,} dim={dim}")
    print(f"[ESTIMATE] embeddings on disk: ~{emb_bytes/1e9:.2f} GB (float32)")
    print(f"[ESTIMATE] final_kb.jsonl metadata: ~{meta_bytes/1e9:.2f} GB")
    print(f"[ESTIMATE] FAISS Flat index (if selected) would need ~{emb_bytes/1e9:.2f} GB additionally")
    return emb_bytes, meta_bytes

_ = estimate_storage(N_TOTAL_RECORDS, EMBED_DIM)

def load_embed_progress():
    p = CONFIG["paths"]["embedding_progress"]
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return {"run_id": RUN_ID, "n_total": N_TOTAL_RECORDS, "dim": EMBED_DIM,
            "representation": CONFIG["embedding"]["representation"],
            "completed_batches": 0, "completed_docs": 0}

def save_embed_progress(progress):
    with open(CONFIG["paths"]["embedding_progress"], "w") as f:
        json.dump(progress, f, indent=2)

def append_embeddings(vecs):
    mode = "ab" if os.path.exists(CONFIG["paths"]["embeddings_memmap"]) else "wb"
    with open(CONFIG["paths"]["embeddings_memmap"], mode) as f:
        f.write(np.ascontiguousarray(vecs, dtype=np.float32).tobytes())

def read_all_embeddings(n_docs, dim):
    return np.memmap(CONFIG["paths"]["embeddings_memmap"], dtype=np.float32, mode="r", shape=(n_docs, dim))


[ESTIMATE] documents=188,152 dim=1024
[ESTIMATE] embeddings on disk: ~0.77 GB (float32)
[ESTIMATE] final_kb.jsonl metadata: ~0.05 GB
[ESTIMATE] FAISS Flat index (if selected) would need ~0.77 GB additionally


## SECTION 17 — Large-Scale Embedding Generation (streams text from disk, resumable) — PATCH 2

In [20]:

# ============================================================
# SECTION 17: EMBEDDING GENERATION LOOP -- reads text straight from final_kb.jsonl
# via iter_kb_records; never builds a Python list of all documents.
# ============================================================
@log_step("Large-scale embedding generation")
def generate_embeddings_resumable():
    progress = load_embed_progress()
    if progress.get("dim") != EMBED_DIM or progress.get("representation") != CONFIG["embedding"]["representation"] \
            or progress.get("n_total") != N_TOTAL_RECORDS:
        print("[EMBED] Config/corpus changed since last checkpoint -> starting fresh embedding file.")
        if os.path.exists(CONFIG["paths"]["embeddings_memmap"]):
            os.remove(CONFIG["paths"]["embeddings_memmap"])
        progress = {"run_id": RUN_ID, "n_total": N_TOTAL_RECORDS, "dim": EMBED_DIM,
                    "representation": CONFIG["embedding"]["representation"],
                    "completed_batches": 0, "completed_docs": 0}
        save_embed_progress(progress)

    bs = CONFIG["embedding"]["batch_size"]
    start_doc = progress["completed_docs"]
    if start_doc >= N_TOTAL_RECORDS:
        print(f"[EMBED] Already complete: {start_doc:,}/{N_TOTAL_RECORDS:,}")
        return progress

    print(f"[EMBED] Resuming from document {start_doc:,} / {N_TOTAL_RECORDS:,}")
    batch_num = progress["completed_batches"]
    t0 = time.time()
    record_stream = iter_kb_records(start=start_doc)
    pos = start_doc
    while pos < N_TOTAL_RECORDS:
        chunk = list(islice(record_stream, bs))
        if not chunk:
            break
        texts = [r["question"] for r in chunk]   # production representation: question-only
        vecs = embed_model.encode(texts, batch_size=bs, normalize_embeddings=CONFIG["embedding"]["normalize"],
                                   show_progress_bar=False, convert_to_numpy=True)
        append_embeddings(vecs)
        pos += len(chunk)
        batch_num += 1
        progress["completed_docs"] = pos
        progress["completed_batches"] = batch_num
        if batch_num % CONFIG["embedding"]["checkpoint_every_batches"] == 0:
            save_embed_progress(progress)
            dt = time.time() - t0
            rate = (pos - start_doc) / max(dt, 1e-6)
            remaining = (N_TOTAL_RECORDS - pos) / max(rate, 1e-6)
            print(f"[EMBED] batch {batch_num} | {pos:,}/{N_TOTAL_RECORDS:,} "
                  f"({rate:.1f} docs/s, ETA {remaining/60:.1f} min) -- checkpoint saved")

    save_embed_progress(progress)
    with open(CONFIG["paths"]["embeddings_meta"], "w") as f:
        json.dump({"n_docs": N_TOTAL_RECORDS, "dim": EMBED_DIM,
                    "representation": CONFIG["embedding"]["representation"],
                    "model": CONFIG["embedding"]["model_name"]}, f, indent=2)
    return progress

EMBED_PROGRESS = generate_embeddings_resumable()
assert EMBED_PROGRESS["completed_docs"] == N_TOTAL_RECORDS, (
    f"[FATAL] Embedding count {EMBED_PROGRESS['completed_docs']} != KB size {N_TOTAL_RECORDS}. "
    f"Re-run this cell to resume.")
print(f"[EMBED] Final progress: {EMBED_PROGRESS}")

KB_EMBEDDINGS = read_all_embeddings(N_TOTAL_RECORDS, EMBED_DIM)
print(f"[EMBED] memmap ready, shape={KB_EMBEDDINGS.shape}")



=== Large-scale embedding generation ===
[EMBED] Resuming from document 0 / 188,152
[EMBED] batch 20 | 1,280/188,152 (177.2 docs/s, ETA 17.6 min) -- checkpoint saved
[EMBED] batch 40 | 2,560/188,152 (179.0 docs/s, ETA 17.3 min) -- checkpoint saved
[EMBED] batch 60 | 3,840/188,152 (183.0 docs/s, ETA 16.8 min) -- checkpoint saved
[EMBED] batch 80 | 5,120/188,152 (186.8 docs/s, ETA 16.3 min) -- checkpoint saved
[EMBED] batch 100 | 6,400/188,152 (189.7 docs/s, ETA 16.0 min) -- checkpoint saved
[EMBED] batch 120 | 7,680/188,152 (180.3 docs/s, ETA 16.7 min) -- checkpoint saved
[EMBED] batch 140 | 8,960/188,152 (184.9 docs/s, ETA 16.2 min) -- checkpoint saved
[EMBED] batch 160 | 10,240/188,152 (187.9 docs/s, ETA 15.8 min) -- checkpoint saved
[EMBED] batch 180 | 11,520/188,152 (191.0 docs/s, ETA 15.4 min) -- checkpoint saved
[EMBED] batch 200 | 12,800/188,152 (191.6 docs/s, ETA 15.3 min) -- checkpoint saved
[EMBED] batch 220 | 14,080/188,152 (192.6 docs/s, ETA 15.1 min) -- checkpoint saved
[E

## SECTION 18 — FAISS Index Selection & Configuration

In [21]:

# ============================================================
# SECTION 18: AUTO-SELECT FAISS INDEX TYPE
# ============================================================
import faiss

def choose_index_type(n_docs, forced=None):
    if forced:
        return forced
    if n_docs < 50_000:
        return "flat"
    if n_docs <= 2_000_000:
        return "hnsw"
    return "ivfpq"

INDEX_TYPE = choose_index_type(N_TOTAL_RECORDS, CONFIG["faiss"]["force_index_type"])
print(f"[FAISS] corpus size={N_TOTAL_RECORDS:,} -> selected index type: {INDEX_TYPE}")

FAISS_CONFIG = {"index_type": INDEX_TYPE, "dim": EMBED_DIM, "n_docs": N_TOTAL_RECORDS,
                 "metric": "inner_product (cosine, vectors L2-normalized)"}
if INDEX_TYPE == "hnsw":
    FAISS_CONFIG.update({"m": CONFIG["faiss"]["hnsw_m"], "ef_construction": CONFIG["faiss"]["hnsw_ef_construction"],
                          "ef_search": CONFIG["faiss"]["hnsw_ef_search"]})
elif INDEX_TYPE == "ivfpq":
    nlist = max(64, int((N_TOTAL_RECORDS / CONFIG["faiss"]["nlist_target_docs_per_cluster"]) ** 0.5) * 4)
    FAISS_CONFIG.update({"nlist": nlist, "pq_bits": CONFIG["faiss"]["pq_bits"], "nprobe": CONFIG["faiss"]["nprobe"]})
print(f"[FAISS] config: {FAISS_CONFIG}")


[FAISS] corpus size=188,152 -> selected index type: hnsw
[FAISS] config: {'index_type': 'hnsw', 'dim': 1024, 'n_docs': 188152, 'metric': 'inner_product (cosine, vectors L2-normalized)', 'm': 32, 'ef_construction': 200, 'ef_search': 128}


## SECTION 19 — Chunked FAISS Index Construction (PATCH 3) + Periodic Checkpoint (PATCH 16)

Vectors are added to FAISS in bounded chunks read from the memmap -- the full multi-million-row array is never copied into a single contiguous buffer. If a build is interrupted partway, restarting reloads the partially built index and continues from the next un-added chunk instead of starting over.

In [22]:

# ============================================================
# SECTION 19: CHUNKED INDEX CONSTRUCTION WITH RESUME
# ============================================================
def _new_empty_index(dim, index_type, cfg, embeddings_for_training=None):
    if index_type == "flat":
        return faiss.IndexFlatIP(dim)
    if index_type == "hnsw":
        idx = faiss.IndexHNSWFlat(dim, cfg["m"], faiss.METRIC_INNER_PRODUCT)
        idx.hnsw.efConstruction = cfg["ef_construction"]
        idx.hnsw.efSearch = cfg["ef_search"]
        return idx
    if index_type == "ivfpq":
        quantizer = faiss.IndexFlatIP(dim)
        m_sub = max(1, dim // CONFIG["faiss"]["pq_m_divisor"])
        idx = faiss.IndexIVFPQ(quantizer, dim, cfg["nlist"], m_sub, cfg["pq_bits"], faiss.METRIC_INNER_PRODUCT)
        train_n = min(len(embeddings_for_training), max(cfg["nlist"] * 40, 100_000))
        train_idx = np.random.choice(len(embeddings_for_training), train_n, replace=False)
        train_idx.sort()  # sorted fancy-indexing is far cheaper on a memmap than random order
        idx.train(np.ascontiguousarray(embeddings_for_training[train_idx], dtype=np.float32))
        idx.nprobe = cfg["nprobe"]
        return idx
    raise ValueError(f"Unknown index type: {index_type}")

@log_step(f"Chunked FAISS index construction")
def build_faiss_index_chunked(embeddings_memmap, n_docs, dim, index_type, cfg):
    build_progress = _load_progress(CONFIG["paths"]["faiss_build_progress"],
                                     {"index_type": None, "chunks_added": 0, "n_docs": n_docs})
    chunk_size = CONFIG["faiss"]["add_chunk_size"]

    resumable = (build_progress.get("index_type") == index_type and build_progress.get("n_docs") == n_docs
                 and os.path.exists(CONFIG["paths"]["faiss_index"]) and build_progress.get("chunks_added", 0) > 0)
    if resumable:
        index = faiss.read_index(CONFIG["paths"]["faiss_index"])
        start_chunk = build_progress["chunks_added"]
        print(f"[FAISS] Resuming build from chunk {start_chunk} ({index.ntotal:,} vectors already added).")
    else:
        index = _new_empty_index(dim, index_type, cfg, embeddings_for_training=embeddings_memmap)
        start_chunk = 0
        build_progress = {"index_type": index_type, "chunks_added": 0, "n_docs": n_docs}

    n_chunks = (n_docs + chunk_size - 1) // chunk_size
    for chunk_i in range(start_chunk, n_chunks):
        lo, hi = chunk_i * chunk_size, min((chunk_i + 1) * chunk_size, n_docs)
        chunk_vecs = np.ascontiguousarray(embeddings_memmap[lo:hi], dtype=np.float32)  # bounded copy, one chunk only
        index.add(chunk_vecs)
        build_progress["chunks_added"] = chunk_i + 1
        print(f"[FAISS] added chunk {chunk_i+1}/{n_chunks} (rows {lo:,}-{hi:,}) -- ntotal={index.ntotal:,}")
        if (chunk_i + 1) % CONFIG["faiss"]["save_every_n_chunks"] == 0 or (chunk_i + 1) == n_chunks:
            faiss.write_index(index, CONFIG["paths"]["faiss_index"])
            _save_progress(CONFIG["paths"]["faiss_build_progress"], build_progress)
            print(f"[FAISS] checkpoint saved at chunk {chunk_i+1}/{n_chunks}")

    assert index.ntotal == n_docs, f"[FATAL] FAISS ntotal {index.ntotal} != corpus size {n_docs}"
    return index

FAISS_INDEX = build_faiss_index_chunked(KB_EMBEDDINGS, N_TOTAL_RECORDS, EMBED_DIM, INDEX_TYPE, FAISS_CONFIG)
faiss.write_index(FAISS_INDEX, CONFIG["paths"]["faiss_index"])
with open(CONFIG["paths"]["faiss_config"], "w") as f:
    json.dump(FAISS_CONFIG, f, indent=2)
print(f"[FAISS] index built and saved: ntotal={FAISS_INDEX.ntotal:,}")
print("[FAISS] NOTE: insertion order == record_id order, so FAISS result position IS the record_id "
      "-- no separate vector-id-to-record dict is needed (PATCH 4).")



=== Chunked FAISS index construction ===
[FAISS] added chunk 1/2 (rows 0-100,000) -- ntotal=100,000
[FAISS] added chunk 2/2 (rows 100,000-188,152) -- ntotal=188,152
[FAISS] checkpoint saved at chunk 2/2
[OK] Chunked FAISS index construction finished in 551.77s
[FAISS] index built and saved: ntotal=188,152
[FAISS] NOTE: insertion order == record_id order, so FAISS result position IS the record_id -- no separate vector-id-to-record dict is needed (PATCH 4).


## SECTION 20 — Index Save/Load Verification + Alignment Assertions (PATCH 20)

In [23]:

# ============================================================
# SECTION 20: RELOAD FROM DISK AND VERIFY END-TO-END ALIGNMENT
# ============================================================
@log_step("FAISS reload + alignment verification")
def reload_and_verify():
    idx = faiss.read_index(CONFIG["paths"]["faiss_index"])
    if idx.ntotal != N_TOTAL_RECORDS:
        raise RuntimeError(f"[FATAL] FAISS ntotal ({idx.ntotal}) != KB size ({N_TOTAL_RECORDS}) after reload.")
    if INDEX_TYPE == "hnsw":
        idx.hnsw.efSearch = FAISS_CONFIG["ef_search"]
    if INDEX_TYPE == "ivfpq":
        idx.nprobe = FAISS_CONFIG["nprobe"]

    probe_vec = np.ascontiguousarray(KB_EMBEDDINGS[0:1], dtype=np.float32)
    D, I = idx.search(probe_vec, 1)
    if I[0][0] != 0:
        raise RuntimeError("[FATAL] Self-retrieval sanity check failed after reload -- "
                            "FAISS position no longer aligns with record_id.")
    resolved = get_record(int(I[0][0]))
    if resolved["record_id"] != 0:
        raise RuntimeError("[FATAL] Resolved record_id does not match FAISS position -- alignment broken.")

    print(f"[FAISS] Reload verified. ntotal={idx.ntotal:,}. "
          f"Self-retrieval + record-id alignment check passed (score={D[0][0]:.4f}).")
    return idx

FAISS_INDEX = reload_and_verify()



=== FAISS reload + alignment verification ===
[FAISS] Reload verified. ntotal=188,152. Self-retrieval + record-id alignment check passed (score=1.0000).
[OK] FAISS reload + alignment verification finished in 1.32s


## SECTION 21 — Retrieval Function (question embedded -> complete QA record) — PATCH 5

In [24]:

# ============================================================
# SECTION 21: RETRIEVAL
# ============================================================
def retrieve(query, top_k=None):
    top_k = top_k or CONFIG["retrieval"]["top_k"]
    q_emb = embed_model.encode([query], normalize_embeddings=CONFIG["embedding"]["normalize"], convert_to_numpy=True)
    q_emb = np.ascontiguousarray(q_emb, dtype=np.float32)
    D, I = FAISS_INDEX.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx == -1:
            continue
        rec = get_record(int(idx))   # FAISS position == record_id (PATCH 4)
        results.append({"record_id": rec["record_id"], "question": rec["question"], "answer": rec["answer"],
                         "source_dataset": rec["source_dataset"], "score": float(score)})
    results.sort(key=lambda r: r["score"], reverse=True)
    return results

def format_records_for_llm(records):
    return "\n\n".join(f"[RECORD {i}]\nQuestion: {r['question']}\nAnswer: {r['answer']}"
                        for i, r in enumerate(records, 1))


## SECTION 22 — Retrieval Testing

In [25]:

# ============================================================
# SECTION 22: QUICK RETRIEVAL SMOKE TEST ON REAL INDEX
# ============================================================
for rec in sample_records(3):
    res = retrieve(rec["question"], top_k=3)
    print(f"\nQuery: {rec['question']}")
    for r in res:
        print(f"   score={r['score']:.3f} [{r['source_dataset']}] Q: {r['question'][:80]}")



Query: When should I irrigate wheat during the tillering stage in [REDACTED], [REDACTED], Uttar Pradesh?
   score=1.000 [farmerchat] Q: When should I irrigate wheat during the tillering stage in [REDACTED], [REDACTED
   score=1.000 [farmerchat] Q: When should I irrigate wheat during the tillering stage in [REDACTED], [REDACTED
   score=1.000 [farmerchat] Q: When should I irrigate wheat during the tillering stage in [REDACTED], [REDACTED

Query: how to grow marigold flower in flower pot easily
   score=1.000 [farmerchat] Q: how to grow marigold flower in flower pot easily
   score=0.848 [farmerchat] Q: how to grow marigold plant from the flower
   score=0.845 [farmerchat] Q: How can I promote good flowering in my marigold plant that is planted in a pot?

Query: finger millets seed rate
   score=1.000 [farmerchat] Q: finger millets seed rate
   score=0.785 [farmerchat] Q: what is the seed rate in pearl millet
   score=0.785 [farmerchat] Q: what is the seed rate in pearl millet


## SECTION 23 — Retrieval Evaluation (Recall@K, MRR, latency)

In [26]:

# ============================================================
# SECTION 23: RETRIEVAL METRICS ON A HELD-OUT / PARAPHRASED EVAL SET (no leakage)
# ============================================================
def make_eval_set(n=500, seed=SEED):
    ids = sample_record_ids(n, seed=seed)
    eval_items = []
    for rid in ids:
        rec = get_record(rid)
        q = rec["question"]
        variant = re.sub(r"^(what|how|when|why|which|where)\s+", "", q.rstrip("?. ").lower())
        if variant != q.lower() and len(variant) >= 5:
            eval_items.append({"query": variant, "gold_record_id": rid})
    return eval_items

EVAL_SET = make_eval_set(n=500)
print(f"[EVAL] retrieval eval set size: {len(EVAL_SET)}")

def evaluate_retrieval(eval_set, top_k_values=(1, 3, 5, 10)):
    max_k = max(top_k_values)
    hits_at = {k: 0 for k in top_k_values}
    rr_sum, latencies = 0.0, []
    for item in eval_set:
        t0 = time.time()
        res = retrieve(item["query"], top_k=max_k)
        latencies.append(time.time() - t0)
        ranked_ids = [r["record_id"] for r in res]
        rank = ranked_ids.index(item["gold_record_id"]) + 1 if item["gold_record_id"] in ranked_ids else None
        for k in top_k_values:
            if rank is not None and rank <= k:
                hits_at[k] += 1
        if rank is not None:
            rr_sum += 1.0 / rank
    n = max(len(eval_set), 1)
    report = {f"recall@{k}": hits_at[k] / n for k in top_k_values}
    report["mrr"] = rr_sum / n
    report["mean_latency_ms"] = 1000 * sum(latencies) / max(len(latencies), 1)
    report["n_eval"] = len(eval_set)
    return report

RETRIEVAL_EVAL_REPORT = evaluate_retrieval(EVAL_SET)
print(f"[EVAL] Retrieval report: {json.dumps(RETRIEVAL_EVAL_REPORT, indent=2)}")


[EVAL] retrieval eval set size: 473
[EVAL] Retrieval report: {
  "recall@1": 0.4630021141649049,
  "recall@3": 0.5348837209302325,
  "recall@5": 0.5708245243128964,
  "recall@10": 0.6067653276955602,
  "mrr": 0.5076168663377967,
  "mean_latency_ms": 33.266488391803634,
  "n_eval": 473
}


## SECTION 23b — Embedding Representation Ablation (OFF by default — PATCH 6)

In [27]:

# ============================================================
# SECTION 23b: QUESTION-ONLY vs QUESTION+ANSWER EMBEDDING (representative subset only)
# ============================================================
if not RUN_EMBEDDING_ABLATION:
    print("[ABLATION] Skipped (RUN_EMBEDDING_ABLATION=False). Enable in Section 1b to run this experiment.")
    EMBEDDING_ABLATION_REPORT = None
else:
    ABLATION_SUBSET_SIZE = min(20_000, N_TOTAL_RECORDS)

    def run_embedding_ablation():
        subset = sample_records(ABLATION_SUBSET_SIZE, seed=SEED + 7)
        id_to_pos = {r["record_id"]: i for i, r in enumerate(subset)}
        sub_eval = []
        for r in subset:
            variant = re.sub(r"^(what|how|when|why|which|where)\s+", "", r["question"].rstrip("?. ").lower())
            if variant != r["question"].lower() and len(variant) >= 5:
                sub_eval.append({"query": variant, "gold_record_id": r["record_id"]})
        sub_eval = sub_eval[:300]
        if not sub_eval:
            print("[ABLATION] Not enough paraphrasable queries -- skipping.")
            return None

        q_vecs = embed_model.encode([r["question"] for r in subset], batch_size=64,
                                     normalize_embeddings=True, show_progress_bar=False)
        qa_vecs = embed_model.encode([f"Question: {r['question']} Answer: {r['answer']}" for r in subset],
                                      batch_size=64, normalize_embeddings=True, show_progress_bar=False)

        def eval_variant(vecs):
            idx = faiss.IndexFlatIP(vecs.shape[1])
            idx.add(np.ascontiguousarray(vecs, dtype=np.float32))
            hits_at, rr_sum = {1: 0, 3: 0, 5: 0}, 0.0
            for item in sub_eval:
                qv = embed_model.encode([item["query"]], normalize_embeddings=True, show_progress_bar=False)
                D, I = idx.search(np.ascontiguousarray(qv, dtype=np.float32), 10)
                gold_pos = id_to_pos[item["gold_record_id"]]
                ranked = list(I[0])
                rank = ranked.index(gold_pos) + 1 if gold_pos in ranked else None
                for k in (1, 3, 5):
                    if rank is not None and rank <= k:
                        hits_at[k] += 1
                if rank is not None:
                    rr_sum += 1.0 / rank
            n = max(len(sub_eval), 1)
            return {f"recall@{k}": hits_at[k]/n for k in (1,3,5)} | {"mrr": rr_sum/n}

        return {"subset_size": ABLATION_SUBSET_SIZE, "eval_size": len(sub_eval),
                "question_only": eval_variant(q_vecs), "question_answer": eval_variant(qa_vecs)}

    EMBEDDING_ABLATION_REPORT = run_embedding_ablation()
    print(f"[ABLATION] {json.dumps(EMBEDDING_ABLATION_REPORT, indent=2)}")
    if EMBEDDING_ABLATION_REPORT:
        qo = EMBEDDING_ABLATION_REPORT["question_only"]["recall@5"]
        qa = EMBEDDING_ABLATION_REPORT["question_answer"]["recall@5"]
        verdict = "question+answer shows a meaningful edge" if qa > qo + 0.02 else "no reliable advantage for question+answer"
        print(f"[ABLATION] {verdict} (recall@5 qo={qo:.3f} vs qa={qa:.3f}). "
              f"Production stays question-only unless you deliberately rebuild the index otherwise.")


[ABLATION] Skipped (RUN_EMBEDDING_ABLATION=False). Enable in Section 1b to run this experiment.


## SECTION 24 — Grounded RAG Prompt (with mandatory refusal path) — PATCH 9

In [28]:

# ============================================================
# SECTION 24: SYSTEM PROMPT + PROMPT BUILDER + GENERATION (with refusal)
# ============================================================
SYSTEM_PROMPT = (
    "You are an agricultural assistant for Indian farmers. You will be given a farmer's "
    "question and several retrieved QA information records from an agricultural knowledge "
    "base. Rules:\n"
    "1. Answer the farmer's question directly, in clear, farmer-friendly language.\n"
    "2. Use the retrieved records as evidence. Synthesize information across multiple "
    "records when the question needs it. Ignore irrelevant records.\n"
    "3. Do NOT invent agricultural facts, dosages, or chemical recommendations that are not "
    "supported by the retrieved records.\n"
    "4. Do NOT claim certainty when the evidence is weak or missing.\n"
    "5. If retrieved records conflict, briefly say the sources differ rather than inventing "
    "a resolution or silently picking one.\n"
    "6. If the retrieved evidence is clearly insufficient, say plainly that you do not have "
    "enough verified information, and do not guess.\n"
    "7. Never mention FAISS, embeddings, retrieval scores, or other internal system details."
)

def build_prompt(question, records):
    context = format_records_for_llm(records) if records else "(no relevant records retrieved)"
    user_msg = (f"FARMER QUESTION:\n{question}\n\nRETRIEVED AGRICULTURAL INFORMATION:\n{context}\n\n"
                "Write a single synthesized, farmer-friendly answer following the system rules.")
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]

def is_low_confidence(records):
    return (not records) or records[0]["score"] < CONFIG["retrieval"]["min_score_for_confidence"]

REFUSAL_MESSAGE = ("I don't have enough verified information in the agricultural knowledge base "
                    "to answer this confidently. Please consult your local agricultural extension "
                    "office or Krishi Vigyan Kendra for this specific case.")

def generate_answer(question, records, max_new_tokens=None):
    max_new_tokens = max_new_tokens or CONFIG["generation"]["max_new_tokens"]
    if is_low_confidence(records):
        return REFUSAL_MESSAGE, True
    msgs = build_prompt(question, records)
    prompt = LLM_TOKENIZER.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = LLM_TOKENIZER(prompt, return_tensors="pt", truncation=True,
                            max_length=CONFIG["generation"]["context_length"]).to(LLM_MODEL.device)
    out = LLM_MODEL.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=CONFIG["generation"]["temperature"], top_p=CONFIG["generation"]["top_p"],
                              pad_token_id=LLM_TOKENIZER.eos_token_id)
    text = LLM_TOKENIZER.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip(), False


## SECTION 25 — End-to-End Pipeline

In [29]:

# ============================================================
# SECTION 25: FULL PIPELINE FUNCTION (models loaded once, reused every call — PATCH 8)
# ============================================================
def krishi_rag_pipeline(question, top_k=None, translate_from_kannada=False):
    timings = {}
    t_start = time.time()
    working_question = question
    if translate_from_kannada and TRANSLATION_READY:
        t0 = time.time(); working_question = translate(question, "src2en"); timings["translation_in_s"] = time.time() - t0
    t0 = time.time(); records = retrieve(working_question, top_k=top_k); timings["retrieval_s"] = time.time() - t0
    t0 = time.time(); answer_en, refused = generate_answer(working_question, records); timings["generation_s"] = time.time() - t0
    final_answer = answer_en
    if translate_from_kannada and TRANSLATION_READY:
        t0 = time.time(); final_answer = translate(answer_en, "en2tgt"); timings["translation_out_s"] = time.time() - t0
    timings["total_s"] = time.time() - t_start
    return {"question": question, "working_question_en": working_question, "retrieved_records": records,
            "refused": refused, "answer": final_answer, "timings": timings}

_demo_q = get_record(0)["question"]
_demo = krishi_rag_pipeline(_demo_q)
print(f"[PIPELINE] demo answer: {_demo['answer'][:300]}")
print(f"[PIPELINE] timings: {_demo['timings']}")


[PIPELINE] demo answer: Crop rotation is very important in farming for several reasons. It helps to maintain soil health and improve its fertility over time. Different crops have different nutrient requirements, so rotating them allows the soil to stay balanced and full of nutrients. This practice also helps to control pes
[PIPELINE] timings: {'retrieval_s': 0.030551671981811523, 'generation_s': 11.780718803405762, 'total_s': 11.811274528503418}


## SECTION 26 — Unseen / Paraphrased Question Tests (always on — cheap, core acceptance tests)

In [30]:

# ============================================================
# SECTION 26: KNOWN vs PARAPHRASED vs UNSEEN vs ABSENT-INFO QUESTIONS
# ============================================================
def run_case(label, question):
    r = krishi_rag_pipeline(question)
    top = r["retrieved_records"][0] if r["retrieved_records"] else None
    print(f"\n--- {label} ---\nQ: {question}")
    if top:
        print(f"Top retrieved: score={top['score']:.3f} | {top['question'][:80]}")
    print(f"Refused: {r['refused']}\nA: {r['answer'][:400]}")
    return r

UNSEEN_TEST_RESULTS = {}
_known_q = get_record(0)["question"]
UNSEEN_TEST_RESULTS["known"] = run_case("Known question (verbatim)", _known_q)

_paraphrase = re.sub(r"^(what|how|when|why|which|where)\s+", "", _known_q.rstrip("?. ").lower())
UNSEEN_TEST_RESULTS["paraphrased"] = run_case("Paraphrased question", _paraphrase)

UNSEEN_TEST_RESULTS["genuinely_unseen"] = run_case(
    "Genuinely unseen composite question",
    "My paddy field is about 30 days old and just received heavy unexpected rainfall. "
    "What should I do about fertilizer application now?")

UNSEEN_TEST_RESULTS["absent_information"] = run_case(
    "Question with likely absent information",
    "What is the exact GPS-coordinate-based subsidy amount for hydroponic saffron "
    "cultivation on my specific 2.3 acre plot?")



--- Known question (verbatim) ---
Q: why is crop rotation important in farming?
Top retrieved: score=1.000 | why is crop rotation important in farming?
Refused: False
A: Crop rotation is very important for farmers because it helps keep the soil healthy and productive. By alternating different crops each growing season, you can:
- Prevent soil erosion and depletion of nutrients
- Reduce pest and disease problems
- Improve soil structure, which helps with water retention and aeration
- Control weeds naturally, reducing the need for chemical weed killers
- Lower the 

--- Paraphrased question ---
Q: is crop rotation important in farming
Top retrieved: score=0.959 | why is crop rotation important in farming?
Refused: False
A: Crop rotation is very important for farmers because it helps keep the soil healthy and productive. By alternating different crops each growing season, you can:
- Maintain soil fertility by preventing the depletion of nutrients, as different crops use and replenish nu

## SECTION 27 — Multi-Record Synthesis Test (always on, single call)

In [31]:

# ============================================================
# SECTION 27: DOES THE LLM COMBINE MULTIPLE RETRIEVED RECORDS?
# ============================================================
MULTI_RECORD_TEST = run_case(
    "Multi-fact synthesis question",
    "My rice crop is at the flowering stage and there has been heavy rain this week -- "
    "what should I consider for fertilizer and crop management right now?")

def rough_groundedness_check(answer, records):
    if not records:
        return None
    answer_low = answer.lower()
    hits = []
    for r in records:
        words = set(re.findall(r"[a-zA-Z]{5,}", r["answer"].lower()))
        if words:
            hits.append(sum(1 for w in words if w in answer_low) / len(words))
    return sum(hits) / len(hits) if hits else None

print(f"[MULTI-RECORD] rough lexical grounding overlap: "
      f"{rough_groundedness_check(MULTI_RECORD_TEST['answer'], MULTI_RECORD_TEST['retrieved_records'])}")



--- Multi-fact synthesis question ---
Q: My rice crop is at the flowering stage and there has been heavy rain this week -- what should I consider for fertilizer and crop management right now?
Top retrieved: score=0.849 | What type of fertilizer should I use during the flowering stage of my rice crop 
Refused: False
A: During the flowering stage of your rice crop, it's important to consider both fertilizer application and water management to ensure optimal growth and yield. Here are key points to follow:

### Fertilization
1. **Use Balanced Fertilizer**: Apply a balanced NPK fertilizer (e.g., 10:26:26) at the beginning of flowering. This helps enhance grain filling and improve yield. Apply around 20-30 kg per he
[MULTI-RECORD] rough lexical grounding overlap: 0.38494615449356534


## SECTION 28 — Top-K Ablation (OFF by default — PATCH 6/10)

In [32]:

# ============================================================
# SECTION 28: TOP-1 / TOP-3 / TOP-5 COMPARISON
# ============================================================
if not RUN_EXPENSIVE_EVALUATION:
    print("[TOPK ABLATION] Skipped (RUN_EXPENSIVE_EVALUATION=False). Enable in Section 1b to run.")
    TOPK_ABLATION_REPORT = None
else:
    TOPK_ABLATION_QUERIES = [item["query"] for item in EVAL_SET[:30]] if EVAL_SET else \
        [get_record(i)["question"] for i in sample_record_ids(30)]

    def topk_retrieval_ablation(queries, ks=(1, 3, 5)):
        report = {}
        for k in ks:
            latencies = []
            for q in queries:
                t0 = time.time(); retrieve(q, top_k=k); latencies.append(time.time() - t0)
            report[f"top_{k}"] = {"mean_latency_ms": 1000 * sum(latencies) / len(latencies)}
        return report

    TOPK_ABLATION_REPORT = topk_retrieval_ablation(TOPK_ABLATION_QUERIES)
    print(f"[TOPK ABLATION] retrieval latency by K: {json.dumps(TOPK_ABLATION_REPORT, indent=2)}")

    _q = MULTI_RECORD_TEST["question"]
    for k in (1, 5):
        recs = retrieve(_q, top_k=k)
        ans, refused = generate_answer(_q, recs)
        print(f"\n[TOPK ABLATION] K={k} refused={refused}\nAnswer: {ans[:300]}")


[TOPK ABLATION] retrieval latency by K: {
  "top_1": {
    "mean_latency_ms": 22.490851084391277
  },
  "top_3": {
    "mean_latency_ms": 20.789678891499836
  },
  "top_5": {
    "mean_latency_ms": 21.166189511617024
  }
}

[TOPK ABLATION] K=1 refused=False
Answer: During the flowering stage of your rice crop, it's crucial to manage your fertilizer and crop carefully. Use a balanced NPK fertilizer with a 10:26:26 ratio (10% nitrogen, 26% phosphorus, and 26% potassium) to support grain development. Apply this fertilizer at the start of flowering to help with gr

[TOPK ABLATION] K=5 refused=False
Answer: During the flowering stage of your rice crop, it's important to balance both fertilizer application and water management. Here are key points to consider:

### Fertilizer Application
- Use a balanced NPK fertilizer with a ratio of 10:26:26. Apply around 20-30 kg per hectare, starting at the beginnin


## SECTION 29 — Generation Evaluation + Source-Wise Analysis

In [33]:

# ============================================================
# SECTION 29: GENERATION QUALITY (semantic similarity, NOT "accuracy") -- gated (many LLM calls)
# ============================================================
def semantic_similarity(a, b):
    va, vb = embed_model.encode([a, b], normalize_embeddings=True, show_progress_bar=False)
    return float(np.dot(va, vb))

if not RUN_EXPENSIVE_EVALUATION:
    print("[GEN-EVAL] Skipped (RUN_EXPENSIVE_EVALUATION=False). Enable in Section 1b to run.")
    GENERATION_EVAL_REPORT, HUMAN_EVAL_TABLE = None, []
else:
    def build_human_eval_table(n=15):
        rows = []
        for rec in sample_records(n, seed=SEED + 1):
            r = krishi_rag_pipeline(rec["question"])
            sim = semantic_similarity(r["answer"], rec["answer"]) if not r["refused"] else None
            rows.append({"question": rec["question"], "reference_answer": rec["answer"][:200],
                         "retrieved_top_score": r["retrieved_records"][0]["score"] if r["retrieved_records"] else None,
                         "generated_answer": r["answer"][:200], "semantic_similarity_to_reference": sim,
                         "refused": r["refused"], "human_correctness": None, "human_relevance": None,
                         "human_completeness": None, "human_flagged_unsupported_content": None})
        return rows

    HUMAN_EVAL_TABLE = build_human_eval_table(n=15)
    sims = [r["semantic_similarity_to_reference"] for r in HUMAN_EVAL_TABLE if r["semantic_similarity_to_reference"] is not None]
    GENERATION_EVAL_REPORT = {"n_evaluated": len(HUMAN_EVAL_TABLE),
                               "mean_semantic_similarity_to_reference": (sum(sims)/len(sims)) if sims else None,
                               "refusal_rate_on_known_questions": sum(1 for r in HUMAN_EVAL_TABLE if r["refused"]) / len(HUMAN_EVAL_TABLE),
                               "note": "Semantic similarity is NOT accuracy. Human columns intentionally blank."}
    print(json.dumps(GENERATION_EVAL_REPORT, indent=2))

    human_eval_csv_path = os.path.join(CONFIG["paths"]["evaluation_dir"], f"{RUN_ID}_human_eval_table.csv")
    with open(human_eval_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(HUMAN_EVAL_TABLE[0].keys()))
        writer.writeheader(); writer.writerows(HUMAN_EVAL_TABLE)
    print(f"[EVAL] human eval table saved -> {human_eval_csv_path}")


{
  "n_evaluated": 15,
  "mean_semantic_similarity_to_reference": 0.916462504863739,
  "refusal_rate_on_known_questions": 0.0,
  "note": "Semantic similarity is NOT accuracy. Human columns intentionally blank."
}
[EVAL] human eval table saved -> /content/drive/MyDrive/majorprojectnlp/evaluation/run_20260830_052920_human_eval_table.csv


In [34]:

# ============================================================
# SECTION 29b: SOURCE-WISE RETRIEVAL ANALYSIS (retrieval-only, cheap, always on)
# ============================================================
def source_wise_eval(eval_set):
    by_source = {}
    for item in eval_set:
        gold_rec = get_record(item["gold_record_id"])
        by_source.setdefault(gold_rec["source_dataset"], []).append(item)
    return {src: evaluate_retrieval(items, top_k_values=(1, 5)) for src, items in by_source.items() if items}

SOURCE_WISE_REPORT = source_wise_eval(EVAL_SET)
print(f"[SOURCE-WISE] {json.dumps(SOURCE_WISE_REPORT, indent=2)}")


[SOURCE-WISE] {
  "farmerchat": {
    "recall@1": 0.45,
    "recall@5": 0.558695652173913,
    "mrr": 0.49036231884057974,
    "mean_latency_ms": 20.9190239077029,
    "n_eval": 460
  },
  "dataset_26k": {
    "recall@1": 1.0,
    "recall@5": 1.0,
    "mrr": 1.0,
    "mean_latency_ms": 20.040603784414436,
    "n_eval": 13
  }
}


## SECTION 30 — Latency Evaluation (OFF by default — PATCH 6)

In [35]:

# ============================================================
# SECTION 30: FULL PIPELINE LATENCY BREAKDOWN (gated -- repeats several full pipeline calls)
# ============================================================
if not RUN_EXPENSIVE_EVALUATION:
    print("[LATENCY] Skipped (RUN_EXPENSIVE_EVALUATION=False). Enable in Section 1b to run.")
    LATENCY_REPORT = None
else:
    LATENCY_TEST_QUERIES = [item["query"] for item in EVAL_SET[:10]] if EVAL_SET else \
        [get_record(i)["question"] for i in sample_record_ids(10)]

    def latency_report(queries):
        agg = {"retrieval_s": [], "generation_s": [], "total_s": []}
        for q in queries:
            r = krishi_rag_pipeline(q)
            for k in agg:
                if k in r["timings"]:
                    agg[k].append(r["timings"][k])
        return {k: {"mean_s": sum(v)/len(v), "max_s": max(v), "min_s": min(v)} for k, v in agg.items() if v}

    LATENCY_REPORT = latency_report(LATENCY_TEST_QUERIES)
    print(f"[LATENCY] {json.dumps(LATENCY_REPORT, indent=2)}")


[LATENCY] {
  "retrieval_s": {
    "mean_s": 0.025509095191955565,
    "max_s": 0.042006731033325195,
    "min_s": 0.020750761032104492
  },
  "generation_s": {
    "mean_s": 17.886594247817992,
    "max_s": 30.91919493675232,
    "min_s": 9.72933316230774
  },
  "total_s": {
    "mean_s": 17.91210608482361,
    "max_s": 30.941224813461304,
    "min_s": 9.771343231201172
  }
}


## SECTION 31 — Interactive NLP Chatbot Test UI (notebook-only)

In [36]:

# ============================================================
# SECTION 31: LIGHTWEIGHT INTERACTIVE TEST LOOP (not a production backend)
# ============================================================
def interactive_test(max_turns=10):
    print("Type a farmer question ('exit' to stop). Max", max_turns, "turns.")
    for _ in range(max_turns):
        try:
            q = input("\nFarmer question: ").strip()
        except EOFError:
            break
        if not q or q.lower() in ("exit", "quit"):
            break
        r = krishi_rag_pipeline(q)
        print("\nRetrieved records:")
        for rec in r["retrieved_records"]:
            print(f"   score={rec['score']:.3f} | {rec['question'][:80]}")
        print(f"\nAnswer: {r['answer']}")
        print(f"Latency: total={r['timings']['total_s']*1000:.0f}ms "
              f"(retrieval={r['timings']['retrieval_s']*1000:.0f}ms, generation={r['timings']['generation_s']*1000:.0f}ms)")

print("[UI] interactive_test() is defined. Call it manually to chat with the pipeline.")


[UI] interactive_test() is defined. Call it manually to chat with the pipeline.


## SECTION 32 — Final Evaluation Report (consolidated)

In [37]:

# ============================================================
# SECTION 32: CONSOLIDATE ALL METRICS (gracefully includes 'skipped: None' for gated sections)
# ============================================================
FINAL_EVAL_REPORT = {
    "run_id": RUN_ID, "run_mode": {"RUN_FULL_CORPUS": RUN_FULL_CORPUS, "RUN_EMBEDDING_ABLATION": RUN_EMBEDDING_ABLATION,
                                    "ENABLE_TRANSLATION": ENABLE_TRANSLATION, "RUN_EXPENSIVE_EVALUATION": RUN_EXPENSIVE_EVALUATION},
    "corpus": PROVENANCE_STATS, "faiss_config": FAISS_CONFIG, "retrieval_eval": RETRIEVAL_EVAL_REPORT,
    "embedding_ablation": EMBEDDING_ABLATION_REPORT, "source_wise_retrieval": SOURCE_WISE_REPORT,
    "topk_ablation": TOPK_ABLATION_REPORT, "generation_eval": GENERATION_EVAL_REPORT, "latency": LATENCY_REPORT,
    "llm_report": LLM_REPORT,
    "unseen_question_tests": {k: {"question": v["question"], "refused": v["refused"], "answer_preview": v["answer"][:200]}
                                for k, v in UNSEEN_TEST_RESULTS.items()},
}
with open(CONFIG["paths"]["eval_report"], "w") as f:
    json.dump(FINAL_EVAL_REPORT, f, indent=2, default=str)
print(f"[REPORT] Saved -> {CONFIG['paths']['eval_report']}")
print(json.dumps({k: ("..." if k == "unseen_question_tests" else v) for k, v in FINAL_EVAL_REPORT.items()}, indent=2, default=str))


[REPORT] Saved -> /content/drive/MyDrive/majorprojectnlp/evaluation/run_20260830_052920_eval_report.json
{
  "run_id": "run_20260830_052920",
  "run_mode": {
    "RUN_FULL_CORPUS": true,
    "RUN_EMBEDDING_ABLATION": false,
    "ENABLE_TRANSLATION": true,
    "RUN_EXPENSIVE_EVALUATION": true
  },
  "corpus": {
    "run_id": "run_20260830_052920",
    "dataset_26k": {
      "source": "dataset_26k",
      "raw": 26088,
      "malformed": 0,
      "missing_q": 0,
      "missing_a": 0,
      "boilerplate": 0,
      "clean": 5804,
      "within_source_duplicates": 20284
    },
    "farmerchat": {
      "source": "farmerchat",
      "raw": 238030,
      "malformed": 0,
      "missing_q": 0,
      "missing_a": 12,
      "boilerplate": 0,
      "clean": 182348,
      "within_source_duplicates": 55670
    },
    "cross_source_duplicates_removed": 0,
    "final_count": 188152,
    "source_breakdown": {
      "dataset_26k": 5804,
      "farmerchat": 182348
    },
    "run_mode": {
      "full_cor

## SECTION 33 — Reproducibility Manifest

In [38]:

# ============================================================
# SECTION 33: FINAL MANIFEST (weights re-downloaded from HF each session; only config/results persist)
# ============================================================
def pkg_versions():
    import importlib
    out = {}
    for m in ["torch", "transformers", "sentence_transformers", "faiss", "accelerate", "sacrebleu", "ijson"]:
        try:
            out[m] = getattr(importlib.import_module(m), "__version__", "unknown")
        except Exception:
            out[m] = "not importable"
    return out

MANIFEST = {
    "run_id": RUN_ID, "timestamp": datetime.now().isoformat(), "seed": SEED, "config": CONFIG,
    "run_mode_flags": {"RUN_FULL_CORPUS": RUN_FULL_CORPUS, "RUN_EMBEDDING_ABLATION": RUN_EMBEDDING_ABLATION,
                        "ENABLE_TRANSLATION": ENABLE_TRANSLATION, "RUN_EXPENSIVE_EVALUATION": RUN_EXPENSIVE_EVALUATION},
    "package_versions": pkg_versions(), "dataset_counts": PROVENANCE_STATS,
    "embedding_model": CONFIG["embedding"]["model_name"], "embedding_dim": EMBED_DIM,
    "faiss_index_type": INDEX_TYPE, "faiss_config": FAISS_CONFIG, "generation_model": LLM_REPORT,
    "translation_ready": TRANSLATION_READY, "paths": CONFIG["paths"],
}
with open(CONFIG["paths"]["manifest"], "w") as f:
    json.dump(MANIFEST, f, indent=2, default=str)
print(f"[MANIFEST] saved -> {CONFIG['paths']['manifest']}")


[MANIFEST] saved -> /content/drive/MyDrive/majorprojectnlp/runs/run_20260830_052920_manifest.json


## SECTION 34 — Final System Health Check (real refusal test — PATCH 12; alignment asserts — PATCH 20)

In [39]:

# ============================================================
# SECTION 34: DEPLOYMENT-READINESS HEALTH CHECK
# ============================================================
def test_refusal_behavior():
    '''PATCH 12: a real check, not `refused in (True, False)`. Sends a deliberately
    unrelated/nonsense query and verifies the system (a) sees low retrieval confidence
    and (b) actually returns the refusal message rather than fabricating an answer.'''
    nonsense_query = "zzxq flibbertigibbet unrelated nonsense nnqqxx 908214"
    records = retrieve(nonsense_query, top_k=5)
    top_score = records[0]["score"] if records else -1.0
    result = krishi_rag_pipeline(nonsense_query)
    low_conf = top_score < CONFIG["retrieval"]["min_score_for_confidence"]
    refused_correctly = result["refused"] is True and result["answer"] == REFUSAL_MESSAGE
    ok = low_conf and refused_correctly
    return ok, {"top_score": top_score, "refused": result["refused"], "answer_preview": result["answer"][:150]}

@log_step("Final system health check")
def run_health_checks():
    checks = []
    def add(name, ok, detail=""):
        checks.append({"check": name, "ok": bool(ok), "detail": str(detail)})

    add("Final KB non-empty", N_TOTAL_RECORDS > 0, f"{N_TOTAL_RECORDS:,} records")
    _probe = sample_records(min(1000, N_TOTAL_RECORDS))
    add("Question<->Answer linkage intact", all("question" in r and "answer" in r for r in _probe))
    add("Embeddings file exists", os.path.exists(CONFIG["paths"]["embeddings_memmap"]))
    add("Embedding count matches KB size (PATCH 20)", EMBED_PROGRESS["completed_docs"] == N_TOTAL_RECORDS,
        f"{EMBED_PROGRESS['completed_docs']:,}/{N_TOTAL_RECORDS:,}")
    add("FAISS index file saved", os.path.exists(CONFIG["paths"]["faiss_index"]))
    add("FAISS ntotal matches KB size (PATCH 20)", FAISS_INDEX.ntotal == N_TOTAL_RECORDS,
        f"faiss={FAISS_INDEX.ntotal:,} kb={N_TOTAL_RECORDS:,}")
    add("Retrieval returns complete QA records",
        all(("question" in r and "answer" in r) for r in retrieve(get_record(0)["question"], top_k=3)))
    add("Retrieval recall@5 reasonable", RETRIEVAL_EVAL_REPORT.get("recall@5", 0) > 0.3,
        f"recall@5={RETRIEVAL_EVAL_REPORT.get('recall@5')}")
    add("LLM loaded and generating", LLM_MODEL is not None)

    refusal_ok, refusal_detail = test_refusal_behavior()
    add("Refusal behavior on nonsense query is REAL, not fabricated (PATCH 12)", refusal_ok, refusal_detail)

    add("Translation status reported", True, f"enabled={CONFIG['translation']['enabled']} ready={TRANSLATION_READY}")
    add("Manifest saved", os.path.exists(CONFIG["paths"]["manifest"]))
    add("Evaluation report saved", os.path.exists(CONFIG["paths"]["eval_report"]))
    add("Provenance stats saved", os.path.exists(CONFIG["paths"]["provenance_stats"]))
    add("GPU utilized if available", (not GPU_AVAILABLE) or (torch.cuda.memory_allocated() > 0))

    n_ok = sum(1 for c in checks if c["ok"])
    print(f"\n{'CHECK':<55}{'STATUS':<10}DETAIL")
    for c in checks:
        print(f"{c['check']:<55}{'PASS' if c['ok'] else 'FAIL':<10}{c['detail']}")
    print(f"\n[HEALTH] {n_ok}/{len(checks)} checks passed.")
    if n_ok != len(checks):
        print("[HEALTH] WARNING: one or more checks failed -- review before relying on this run.")
    return checks

HEALTH_CHECKS = run_health_checks()
with open(os.path.join(CONFIG["paths"]["evaluation_dir"], f"{RUN_ID}_health_checks.json"), "w") as f:
    json.dump(HEALTH_CHECKS, f, indent=2)



=== Final system health check ===

CHECK                                                  STATUS    DETAIL
Final KB non-empty                                     PASS      188,152 records
Question<->Answer linkage intact                       PASS      
Embeddings file exists                                 PASS      
Embedding count matches KB size (PATCH 20)             PASS      188,152/188,152
FAISS index file saved                                 PASS      
FAISS ntotal matches KB size (PATCH 20)                PASS      faiss=188,152 kb=188,152
Retrieval returns complete QA records                  PASS      
Retrieval recall@5 reasonable                          PASS      recall@5=0.5708245243128964
LLM loaded and generating                              PASS      
Refusal behavior on nonsense query is REAL, not fabricated (PATCH 12)FAIL      {'top_score': 0.5185359716415405, 'refused': False, 'answer_preview': 'Hello! It seems there might be a mix-up with your question. Based o

In [ ]:
interactive_test()

Type a farmer question ('exit' to stop). Max 10 turns.
